# 🗺️ SafeRoam — AI Tourist, Itinerary & Safety System (Colab)

One app: the SafeRoam dashboard + the Groq budget itinerary planner, driven by live data.

**New in this version:** a real interactive map (OpenStreetMap tiles, safety pins, hospitals, your location), live search suggestions for any destination, and a redesigned itinerary (budget meter, day-by-day timeline) you can download as **PDF, HTML or Markdown**.

**Setup (once):** get a free key at console.groq.com → in Colab click 🔑 **Secrets** → add `GROQ_API_KEY` → enable *Notebook access*.
Then run cell 1, then cell 2. If the embedded app looks blank, open the printed link in a new tab (also needed for GPS).

In [1]:
!pip -q install groq

In [2]:
# =============================================================================
# SafeRoam — AI Tourist, Itinerary & Safety System (single-file, Colab-ready)
#
# Merges:  (1) the SafeRoam React dashboard  +  (2) the Groq budget itinerary planner
# Backend: Flask  (Groq compound web-search, Open-Meteo weather, OpenStreetMap lookups)
# Frontend: the same dark SafeRoam UI, now driven by live data (live Leaflet map, typeahead search,
#           structured itinerary view with PDF / HTML / Markdown download).
#
# Colab:  !pip -q install groq   ->  run this cell  ->  open the app link / iframe.
# =============================================================================

import os, re, json, math, time, uuid, threading
import requests
from flask import Flask, request, jsonify, Response
from groq import Groq

PORT = 5055
COMPOUND_MODEL = "groq/compound-mini"   # server-side web search (fast, single tool call)
FAST_MODEL = "openai/gpt-oss-20b"       # plain model: extraction, chat fallback, itinerary fallback
UA = {"User-Agent": "SafeRoam-Colab/1.0 (student project)"}


# ------------------------------------------------------------------ API key --
def get_api_key() -> str:
    """Env var -> Colab Secrets ('GROQ_API_KEY') -> hidden prompt. Never hardcode keys."""
    key = os.getenv("GROQ_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata
        key = userdata.get("GROQ_API_KEY")
        if key:
            return key
    except Exception:
        pass
    import getpass
    return getpass.getpass("Enter your Groq API key: ").strip()


GROQ_API_KEY = get_api_key()
if not GROQ_API_KEY:
    raise RuntimeError("No Groq API key provided.")
client = Groq(api_key=GROQ_API_KEY, timeout=45.0)


# --------------------------------------------------------------- LLM helpers --
def _text(completion) -> str:
    return (completion.choices[0].message.content or "").strip()


def compound(messages, model=COMPOUND_MODEL, max_tokens=4096, temperature=0.4, timeout=45.0) -> str:
    """Groq compound (real web search). Native SDK: LangChain's tool_choice='none' breaks it."""
    c = client.with_options(timeout=timeout).chat.completions.create(
        model=model, temperature=temperature, max_tokens=max_tokens, messages=messages)
    return _text(c)


def fast(messages, max_tokens=4096, temperature=0.5, timeout=45.0) -> str:
    c = client.with_options(timeout=timeout).chat.completions.create(
        model=FAST_MODEL, temperature=temperature, max_tokens=max_tokens, messages=messages,
        extra_body={"reasoning_effort": "low"})
    return _text(c)


def extract_json(text):
    """Pull the first JSON array/object out of an LLM reply (handles ``` fences / chatter)."""
    if not text:
        return None
    text = re.sub(r"```(?:json)?", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    starts = [i for i in (text.find("["), text.find("{")) if i != -1]
    if not starts:
        return None
    s = min(starts)
    closer = "]" if text[s] == "[" else "}"
    e = text.rfind(closer)
    if e <= s:
        return None
    try:
        return json.loads(text[s:e + 1])
    except Exception:
        return None


# ------------------------------------------------------------ misc utilities --
def haversine(lat1, lon1, lat2, lon2) -> float:
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def http_json(url, params=None, timeout=12):
    r = requests.get(url, params=params, headers=UA, timeout=timeout)
    r.raise_for_status()
    return r.json()


_CACHE, _CACHE_TTL = {}, 6 * 3600


def cache_get(key):
    v = _CACHE.get(key)
    return v[1] if v and time.time() - v[0] < _CACHE_TTL else None


def cache_set(key, value):
    _CACHE[key] = (time.time(), value)
    return value


_nom_lock, _nom_last = threading.Lock(), [0.0]


def nominatim(params):
    """OpenStreetMap Nominatim, throttled to <= 1 request/second (their usage policy)."""
    with _nom_lock:
        wait = 1.1 - (time.time() - _nom_last[0])
        if wait > 0:
            time.sleep(wait)
        try:
            return http_json("https://nominatim.openstreetmap.org/search", params)
        finally:
            _nom_last[0] = time.time()


# ---------------------------------------------------------------- background --
JOBS = {}


def start_job(fn, *args):
    """Slow LLM work runs in a thread; the browser polls /api/job/<id> (avoids proxy timeouts)."""
    now = time.time()
    for k in [k for k, v in JOBS.items() if now - v["ts"] > 3600]:
        JOBS.pop(k, None)
    jid = uuid.uuid4().hex[:12]
    JOBS[jid] = {"status": "running", "ts": now}

    def runner():
        try:
            JOBS[jid] = {"status": "done", "result": fn(*args), "ts": time.time()}
        except Exception as e:
            JOBS[jid] = {"status": "error", "error": str(e) or e.__class__.__name__, "ts": time.time()}

    threading.Thread(target=runner, daemon=True).start()
    return jid


# ----------------------------------------------------------------- geocoding --
def geocode(query: str) -> dict:
    query = (query or "").strip()
    if not query:
        raise ValueError("Please enter a city name.")
    parts = [p.strip() for p in query.split(",") if p.strip()]
    name, hint = parts[0], " ".join(parts[1:]).lower()
    try:
        data = http_json("https://geocoding-api.open-meteo.com/v1/search",
                         {"name": name, "count": 10, "language": "en", "format": "json"})
        res = data.get("results") or []
        if res:
            pick = res[0]
            if hint:
                for r in res:
                    hay = " ".join(str(r.get(k, "")) for k in ("country", "country_code", "admin1", "admin2")).lower()
                    if hint in hay:
                        pick = r
                        break
            country = pick.get("country", "")
            return {"name": pick["name"], "country": country,
                    "country_code": (pick.get("country_code") or "").upper(),
                    "admin1": pick.get("admin1", ""), "lat": pick["latitude"], "lon": pick["longitude"],
                    "label": f"{pick['name']}, {country}" if country else pick["name"]}
    except Exception:
        pass
    data = nominatim({"q": query, "format": "jsonv2", "limit": 1, "addressdetails": 1})
    if not data:
        raise ValueError(f"Couldn't find \u201c{query}\u201d. Try a city name like 'Jaipur' or 'Paris, France'.")
    r = data[0]
    addr = r.get("address", {})
    nm = r.get("name") or r.get("display_name", query).split(",")[0]
    country = addr.get("country", "")
    return {"name": nm, "country": country, "country_code": (addr.get("country_code") or "").upper(),
            "admin1": addr.get("state", ""), "lat": float(r["lat"]), "lon": float(r["lon"]),
            "label": f"{nm}, {country}" if country else nm}


# -------------------------------------------------------------- autocomplete --
def _om_place(r) -> dict:
    country = r.get("country", "") or ""
    admin1 = r.get("admin1", "") or ""
    return {"name": r["name"], "country": country, "country_code": (r.get("country_code") or "").upper(),
            "admin1": admin1, "lat": r["latitude"], "lon": r["longitude"],
            "label": f"{r['name']}, {country}" if country else r["name"],
            "sub": ", ".join(x for x in (admin1, country) if x)}


def _osm_place(r) -> dict:
    addr = r.get("address", {}) or {}
    nm = r.get("name") or str(r.get("display_name", "")).split(",")[0]
    country, state = addr.get("country", ""), addr.get("state", "")
    return {"name": nm, "country": country, "country_code": (addr.get("country_code") or "").upper(),
            "admin1": state, "lat": float(r["lat"]), "lon": float(r["lon"]),
            "label": f"{nm}, {country}" if country else nm,
            "sub": ", ".join(x for x in (state, country) if x)}


def suggest_places(q: str) -> list:
    """Typeahead for the destination boxes: Open-Meteo first (fast), OpenStreetMap to fill gaps."""
    q = (q or "").strip()
    if len(q) < 2:
        return []
    key = ("suggest", q.lower())
    hit = cache_get(key)
    if hit is not None:
        return hit
    parts = [p.strip() for p in q.split(",") if p.strip()]
    name = parts[0] if parts else q
    hint = " ".join(parts[1:]).lower()
    out, seen = [], set()

    def add(p):
        sig = (p["name"].lower(), round(p["lat"], 1), round(p["lon"], 1))
        if sig not in seen:
            seen.add(sig)
            out.append(p)

    try:
        data = http_json("https://geocoding-api.open-meteo.com/v1/search",
                         {"name": name, "count": 8, "language": "en", "format": "json"}, timeout=6)
        for r in data.get("results") or []:
            add(_om_place(r))
    except Exception:
        pass
    if hint:  # "Paris, Texas" -> put matches for the hint first
        out.sort(key=lambda p: 0 if hint in " ".join([p["country"], p["country_code"], p["admin1"]]).lower() else 1)
    if len(out) < 3 and len(q) >= 3:  # landmarks / small places Open-Meteo doesn't know
        try:
            for r in nominatim({"q": q, "format": "jsonv2", "limit": 5, "addressdetails": 1}) or []:
                add(_osm_place(r))
        except Exception:
            pass
    out = out[:8]
    if out:
        cache_set(key, out)
    return out


# ------------------------------------------------------------------- weather --
def _wmo(code):
    if code == 0: return "Clear sky", "\u2600\ufe0f"
    if code == 1: return "Mainly clear", "\U0001F324\ufe0f"
    if code == 2: return "Partly cloudy", "\u26c5"
    if code == 3: return "Overcast", "\u2601\ufe0f"
    if code in (45, 48): return "Foggy", "\U0001F32B\ufe0f"
    if 51 <= code <= 57: return "Drizzle", "\U0001F326\ufe0f"
    if 61 <= code <= 67: return "Rain", "\U0001F327\ufe0f"
    if 71 <= code <= 77 or code in (85, 86): return "Snow", "\U0001F328\ufe0f"
    if 80 <= code <= 82: return "Rain showers", "\U0001F326\ufe0f"
    if code >= 95: return "Thunderstorm", "\u26c8\ufe0f"
    return "Cloudy", "\u2601\ufe0f"


def get_weather(lat, lon) -> dict:
    d = http_json("https://api.open-meteo.com/v1/forecast", {
        "latitude": lat, "longitude": lon, "timezone": "auto",
        "current": "temperature_2m,relative_humidity_2m,apparent_temperature,weather_code,wind_speed_10m"})
    c = d["current"]
    desc, icon = _wmo(int(c.get("weather_code", 0)))
    t, feels = c["temperature_2m"], c.get("apparent_temperature", c["temperature_2m"])
    code = int(c.get("weather_code", 0))
    if feels >= 41: adv = "Extreme heat \u2014 avoid midday outdoors, drink water often"
    elif feels >= 35: adv = "High heat \u2014 carry water bottles"
    elif code >= 95: adv = "Thunderstorms \u2014 stay indoors until it passes"
    elif code >= 51 and code not in (71, 73, 75, 77, 85, 86): adv = "Rain likely \u2014 carry an umbrella"
    elif feels <= 5: adv = "Very cold \u2014 dress in warm layers"
    else: adv = "Comfortable conditions for sightseeing"
    return {"temp": round(t), "feels": round(feels), "humidity": round(c["relative_humidity_2m"]),
            "wind": round(c["wind_speed_10m"]), "desc": desc, "icon": icon, "advisory": adv,
            "hot": feels >= 35}


# ------------------------------------------------------------------- places --
SAFETY_LEVELS = ("safe", "moderate", "caution")

DELHI_SEED = [  # used only if the AI lookup fails for Delhi (the original demo data, real coordinates)
    {"name": "India Gate", "area": "Central Delhi", "lat": 28.6129, "lon": 77.2295, "safety": "safe", "rating": 4.8, "tags": ["Monument", "Open-air"]},
    {"name": "Red Fort", "area": "Old Delhi", "lat": 28.6562, "lon": 77.2410, "safety": "safe", "rating": 4.7, "tags": ["History", "Heritage"]},
    {"name": "Lotus Temple", "area": "Kalkaji", "lat": 28.5535, "lon": 77.2588, "safety": "safe", "rating": 4.7, "tags": ["Spiritual", "Architecture"]},
    {"name": "Qutub Minar", "area": "Mehrauli", "lat": 28.5245, "lon": 77.1855, "safety": "moderate", "rating": 4.6, "tags": ["Heritage", "UNESCO"]},
    {"name": "Humayun's Tomb", "area": "Nizamuddin", "lat": 28.5933, "lon": 77.2507, "safety": "safe", "rating": 4.5, "tags": ["Heritage", "Gardens"]},
    {"name": "Jama Masjid", "area": "Old Delhi", "lat": 28.6507, "lon": 77.2334, "safety": "caution", "rating": 4.4, "tags": ["Religious", "Culture"]},
    {"name": "Chandni Chowk", "area": "Old Delhi", "lat": 28.6506, "lon": 77.2303, "safety": "caution", "rating": 4.3, "tags": ["Market", "Street Food"]},
]


def clean_places(items, center) -> list:
    if isinstance(items, dict):
        items = next((v for v in items.values() if isinstance(v, list)), [])
    out, seen = [], set()
    for it in items or []:
        if not isinstance(it, dict):
            continue
        name = str(it.get("name", "")).strip()
        if not name or name.lower() in seen:
            continue
        try:
            lat, lon = float(it.get("lat")), float(it.get("lon"))
            ok = (-90 <= lat <= 90 and -180 <= lon <= 180
                  and haversine(lat, lon, center["lat"], center["lon"]) <= 60)
        except (TypeError, ValueError):
            lat = lon = None
            ok = False
        if not ok:  # model coordinates missing/implausible -> verify against OpenStreetMap
            try:
                g = nominatim({"q": f"{name}, {center['name']}", "format": "jsonv2", "limit": 1})
                if g and haversine(float(g[0]["lat"]), float(g[0]["lon"]), center["lat"], center["lon"]) <= 60:
                    lat, lon = float(g[0]["lat"]), float(g[0]["lon"])
                else:
                    continue
            except Exception:
                continue
        safety = str(it.get("safety", "moderate")).lower().strip()
        safety = safety if safety in SAFETY_LEVELS else "moderate"
        try:
            rating = round(min(5.0, max(1.0, float(it.get("rating", 4.3)))), 1)
        except (TypeError, ValueError):
            rating = 4.3
        tags = it.get("tags") if isinstance(it.get("tags"), list) else []
        out.append({"name": name[:60], "area": str(it.get("area", ""))[:40], "lat": lat, "lon": lon,
                    "safety": safety, "rating": rating, "tags": [str(t)[:22] for t in tags][:3]})
        seen.add(name.lower())
        if len(out) >= 8:
            break
    for i, p in enumerate(out, 1):
        p["id"] = i
    return out


def build_places(c) -> dict:
    key = ("places", c["label"].lower())
    if cache_get(key):
        return cache_get(key)
    base = (
        f"List 8 of the most popular, well-known tourist attractions in {c['label']} that a first-time "
        f"visitor would want to see. Every place MUST physically be in or right next to {c['name']}. "
        "Return ONLY a JSON array (no prose, no markdown) of 8 objects with exactly these keys: "
        '"name" (string), "area" (neighbourhood, string), "lat" (decimal number), "lon" (decimal number), '
        '"safety" (one of "safe", "moderate", "caution" \u2014 your honest estimate for tourists based on '
        'crowding, petty-crime/scam reports and traveller advisories around that spot), '
        '"rating" (typical tourist-review rating 1-5, one decimal), "tags" (2-3 short strings).'
    )
    items = None
    try:
        items = extract_json(compound([{"role": "user", "content": "Search the web for accurate, current info. " + base}], max_tokens=3000))
    except Exception:
        pass
    if not items:
        try:
            items = extract_json(fast([{"role": "user", "content": base}], max_tokens=3000, temperature=0.3))
        except Exception:
            items = None
    places = clean_places(items, c) if items else []
    if not places and c["name"].lower() in ("delhi", "new delhi"):
        places = [dict(p, id=i) for i, p in enumerate(DELHI_SEED, 1)]
    if not places:
        raise RuntimeError(f"Couldn't load attractions for {c['label']} right now. Please try again.")
    return cache_set(key, {"places": places})


# ------------------------------------------------------------------- safety --
_EU = ("DE FR ES IT PT NL BE AT IE GR PL CZ SE DK FI HU RO BG HR SK SI LT LV EE LU MT CY").split()
EMERGENCY_TABLE = {
    "IN": {"police": "112 (or 100)", "ambulance": "108", "fire": "101", "tourist": "1800-111-363"},
    "US": {"police": "911", "ambulance": "911", "fire": "911"},
    "CA": {"police": "911", "ambulance": "911", "fire": "911"},
    "GB": {"police": "999", "ambulance": "999", "fire": "999"},
    "AU": {"police": "000", "ambulance": "000", "fire": "000"},
    "NZ": {"police": "111", "ambulance": "111", "fire": "111"},
    **{cc: {"police": "112", "ambulance": "112", "fire": "112"} for cc in _EU},
}


def build_safety(c) -> dict:
    key = ("safety", c["label"].lower())
    if cache_get(key):
        return cache_get(key)
    prompt = (
        f"Search the web for current travel-safety advisories and typical tourist risks in {c['label']}. "
        "Return ONLY a JSON object (no prose, no markdown) with keys: "
        '"emergency": {"police": str, "ambulance": str, "fire": str, "tourist": str or null} '
        f"(official emergency numbers for {c['country'] or c['name']}), "
        '"alerts": array of 3-4 objects {"title": str (max 8 words), "desc": str (max 18 words, practical), '
        '"type": "safe"|"moderate"|"caution"} covering the most relevant current risks (scams, crowds, heat/weather, '
        "transport, areas to avoid at night) and, if true, one reassurance. Be specific to this city."
    )
    data = None
    try:
        data = extract_json(compound([{"role": "user", "content": prompt}], max_tokens=1500))
    except Exception:
        pass
    if not isinstance(data, dict):
        try:
            data = extract_json(fast([{"role": "user", "content": prompt.replace("Search the web for", "Using your knowledge of")}], max_tokens=1200, temperature=0.3))
        except Exception:
            data = None
    data = data if isinstance(data, dict) else {}
    em = {k: str(v) for k, v in (data.get("emergency") or {}).items()
          if k in ("police", "ambulance", "fire", "tourist") and v}
    em.update(EMERGENCY_TABLE.get(c["country_code"], {}))  # verified table wins over model output
    alerts = []
    for a in (data.get("alerts") or [])[:4]:
        if isinstance(a, dict) and a.get("title"):
            t = str(a.get("type", "moderate")).lower()
            alerts.append({"title": str(a["title"])[:70], "desc": str(a.get("desc", ""))[:140],
                           "type": t if t in SAFETY_LEVELS else "moderate"})
    return cache_set(key, {"emergency": em, "alerts": alerts})


def get_hospitals(lat, lon) -> list:
    key = ("hosp", round(lat, 2), round(lon, 2))
    if cache_get(key) is not None:
        return cache_get(key)
    d = 0.2
    data = nominatim({"q": "hospital", "format": "jsonv2", "limit": 15, "bounded": 1,
                      "viewbox": f"{lon - d},{lat + d},{lon + d},{lat - d}"})
    hosp = []
    for r in data or []:
        nm = r.get("name") or str(r.get("display_name", "")).split(",")[0]
        if nm:
            hosp.append({"name": nm[:60], "lat": float(r["lat"]), "lon": float(r["lon"]),
                         "dist": round(haversine(lat, lon, float(r["lat"]), float(r["lon"])), 1)})
    hosp.sort(key=lambda h: h["dist"])
    return cache_set(key, hosp[:3])


# --------------------------------------------------------------------- chat --
def do_chat(d) -> dict:
    ctx = d.get("context") or {}
    sel = ctx.get("selected")
    lines = [f"City: {ctx.get('city', 'unknown')}."]
    if sel:
        lines.append(f"The user selected: {sel.get('name')} ({sel.get('area') or 'n/a'}), AI-estimated safety: {sel.get('safety')}.")
    if ctx.get("weather"):
        lines.append(f"Current weather: {ctx['weather']}.")
    if ctx.get("emergency"):
        lines.append("Official emergency numbers: " + ", ".join(f"{k} {v}" for k, v in ctx["emergency"].items()) + ".")
    system = (
        "You are SafeRoam's AI travel & safety guide inside a tourist dashboard. " + " ".join(lines) + " "
        "Rules: keep answers under 110 words; use **bold** only for place/dish names or key facts; no headings, tables "
        "or other markdown (use lines starting with \u2022 for short lists); be practical about safety, food, hotels, "
        "transport, routes and costs; NEVER invent phone numbers \u2014 only quote the emergency numbers above; if you are "
        "unsure of a price, hour or fact, say to verify locally. For any real emergency, tell them to call the local "
        "emergency number immediately."
    )
    hist = [{"role": "assistant" if m.get("role") == "assistant" else "user", "content": str(m.get("content", ""))[:1500]}
            for m in (d.get("messages") or [])[-8:]]
    while hist and hist[0]["role"] == "assistant":
        hist.pop(0)
    if not hist:
        raise ValueError("Empty message.")
    msgs = [{"role": "system", "content": system}] + hist
    reply = ""
    try:
        reply = compound(msgs, max_tokens=700, timeout=30.0)
    except Exception:
        pass
    if not reply:
        reply = fast(msgs, max_tokens=700, timeout=30.0)
    return {"reply": reply or "Sorry, I couldn't come up with an answer \u2014 please try again."}


# ---------------------------------------------------------------- itinerary --
_ITINERARY_REQUIREMENTS = """Requirements:
1. Organize the plan day-by-day (Day 1, Day 2, ...), with a morning / afternoon / evening structure and approximate timings.
2. For each activity, include the specific place name and a REALISTIC estimated cost in {currency} PER PERSON (state clearly if a price is per-person or for the whole group, if the budget implies more than one traveler). Search for and use the actual real price of the specific named place wherever possible; do not default to round, made-up numbers like 0, 50, 100, 150, or 200 out of habit, and do not reuse the same figure for multiple unrelated items. Vary prices realistically based on venue type:
   - Street food / snack stall: roughly 20-80 in {currency}-equivalent terms for a small city, scaled up for expensive cities.
   - Casual local sit-down meal (thali, dhaba, cafe): a genuine full meal here, not a snack, and should cost noticeably more than a street snack - typically 2-4x a street snack's price.
   - Mid-range restaurant meal: noticeably more again, and should reflect what a tourist review site or menu would actually show for that establishment.
   - Upscale/fine dining or a well-known specialty restaurant: price it accordingly high - don't undercut it to match cheaper meals elsewhere in the same itinerary.
   - Bars, lounges, pubs, and nightlife venues (including cafes with live music/cover charge): alcoholic and specialty drinks (cocktails, craft beer, wine) are priced ABOVE regular meal items, not below or equal to them - typically several times the cost of a casual meal item, and often with an additional cover charge or minimum spend for live music/entertainment venues. A single drink at this kind of venue should rarely be priced the same as a cheap street snack; if the traveler is likely to have more than one drink over an evening, reflect that in the total, not just a single-drink price labeled as the whole visit's cost.
   Treat the above as illustrative anchors only, not literal numbers - replace them with real prices for {city} specifically (a snack in an expensive global city costs far more than one in a budget destination). Major monuments, palaces, forts, and museums in tourist cities usually charge a real entry fee (often higher for foreign tourists than locals, and sometimes with an extra camera fee) - only mark something as free if it genuinely has no entry fee. Guided tours and private transport (cabs, tuk-tuks/rickshaws for real distances) should reflect typical market rates in that city, not token amounts.
3. For travel BETWEEN locations, give the real approximate distance and time by the most sensible mode of transport - do not default to labeling everything a 'short walk'. If two places are more than about 1-1.5 km apart, or if the terrain is uphill/rough, assume the traveler needs an auto-rickshaw, taxi, bus, or other local transport rather than walking, and price that transport realistically. If you are not confident about the exact distance or time between two specific places, say so explicitly (e.g., 'approx. 2-3 hrs by road, verify locally') rather than guessing a short, confident-sounding number.
   TRANSPORT FARES MUST SCALE WITH DISTANCE - never reuse the same flat fare (e.g. always 50, always 15) for every ride regardless of how far it actually is. Work it out properly for {city}:
   - Metro/train/bus: these systems charge by distance slab or zone, not a single flat citywide fare - a short 1-2 stop hop costs less than crossing several zones or ~7+ km. Use the real fare structure for {city}'s actual transit system if you know it, and make the number go up for longer trips within the same itinerary.
   - Auto-rickshaw/taxi/rideshare: these are priced per kilometer (plus a base/minimum fare), so a 2 km ride and a 10 km ride must NOT show the same cost. Estimate using a realistic per-km rate for {city} and multiply by the actual distance for that specific leg, adding the base fare.
   If two transport legs in the same itinerary have visibly different distances, their costs must visibly differ too - a repeated identical number across multiple different-distance rides is a sign you defaulted to a placeholder instead of calculating it, and should be corrected.
4. Suggest budget-friendly options first (discounted combo tickets, local eateries, public transport, city tourist passes) and clearly flag any splurge-worthy items as optional - but keep even the budget options priced realistically rather than artificially cheap.
5. Group nearby locations on the same day to minimize transport time and cost, but never sacrifice realism about distance/time just to make a tight schedule look feasible.
6. At the end of EACH day, give a running subtotal for that day.
7. At the very end, give a full trip cost breakdown table (Lodging, Food, Transport, Activities, Misc/Buffer) that sums to roughly the stated budget, and explicitly note whether the plan is under, at, or over budget - if the realistic costs exceed the stated budget, say so plainly and suggest concrete ways to cut costs (fewer paid attractions, cheaper lodging tier, more public transport) rather than silently lowering prices to force a fit.
8. Include 2-3 practical money-saving tips specific to {city}.
9. If you are not confident about an exact current price, distance, or time, give a realistic estimated range (e.g. '150-250' or '2-3 hrs') rather than a suspiciously precise or suspiciously round single figure.
10. TABLE FORMATTING: if you use a Markdown table (e.g. for a transport summary), every data row MUST have exactly the same number of pipe-separated cells as the header row, with each distinct piece of information in its own cell. Do NOT combine two fields into one cell - for a 'From'/'To' pair, put the origin in the 'From' cell and the destination in the 'To' cell as two separate cells, not as a single 'Origin -> Destination' string in one cell. A mismatched cell count shifts every later column out of alignment (e.g. distance appearing under the 'Mode' header, or cost appearing under 'Distance'), which makes the whole table misleading - double check that each row's values line up under the correct header before finalizing.
11. GEOGRAPHIC ACCURACY: {city} may name a single destination or multiple destinations (e.g. 'Pune, Nashik' for a multi-city trip). Every place, restaurant, or venue you name MUST actually be located in one of the specified destination(s) - never include a venue from a different city, even if a same-named chain/brand also has an outlet elsewhere (e.g. do not pull in a Mumbai branch of a chain when the trip is to Pune). If multiple cities were specified, clearly assign each day (or block of days) to ONE specific city, note the inter-city travel between them (mode, realistic duration, and cost) as its own itinerary item, and do not mix venues from different cities within the same day unless they are genuinely in the same metro area.
12. INTEREST COVERAGE: the traveler's interests are: {interests}. EVERY interest listed there must be represented by at least one concrete, named activity somewhere in the itinerary - do not silently drop one because the schedule feels full. For each interest, prioritize the CITY'S ACTUAL well-known, popular, highly-rated attractions in that category over generic or obscure ones - for example, if 'adventure' is listed for a city with a famous viewpoint, zipline, trek, or adventure park that tourists commonly visit, name that specific well-known spot rather than inventing a generic 'adventure activity'; if 'temples' is listed, include the city's most notable/most-visited temple(s), not just any temple; the same logic applies to every interest category given. Search for or recall the specific real names of these landmark attractions for {city} rather than describing the activity type abstractly. If a 'Known highlights' list is provided below, treat those named places as strong candidates to include - work them into the itinerary unless they genuinely don't fit the schedule or budget, in which case briefly note why one was skipped. If truly not enough time exists to cover every interest across all {days} day(s), say so explicitly at the end rather than quietly omitting an interest without comment."""

ITINERARY_PROMPT = (
    "You are an expert budget travel planner. You have access to real-time web search - USE IT to look up "
    "actual current entry fees, typical meal prices, and realistic travel distances/times for specific places in "
    "{city} rather than guessing from memory, especially for well-known attractions where verifiable info is "
    "likely available online. Create a detailed {days}-day itinerary for {city} for a traveler with a TOTAL trip "
    "budget of {budget} {currency} (covering lodging, food, local transport, and activities - NOT international "
    "flights). Their interests are: {interests}.\n\n{highlights_block}" + _ITINERARY_REQUIREMENTS)

# The plain fallback model has NO tools - its prompt must not claim web-search ability.
ITINERARY_PROMPT_FALLBACK = (
    "You are an expert budget travel planner. Using your best general knowledge (you do NOT have internet or web "
    "search access for this request, so do not attempt to browse or search - rely only on what you already know), "
    "create a detailed {days}-day itinerary for {city} for a traveler with a TOTAL trip budget of {budget} "
    "{currency} (covering lodging, food, local transport, and activities - NOT international flights). Their "
    "interests are: {interests}.\n\n{highlights_block}" + _ITINERARY_REQUIREMENTS)


def fetch_highlights(city, interests) -> str:
    """Grounding step: real named attractions per interest, so famous spots aren't missed."""
    q = (f"List the most famous, popular, highly-rated named tourist attractions in {city} for each of these "
         f"traveler interests: {', '.join(interests)}. For each interest, give 2-3 SPECIFIC real place names (not "
         f"generic descriptions) that are genuinely well-known must-see spots in {city} for that category. "
         f"Format as:\nInterest: Place Name 1, Place Name 2, Place Name 3\nKeep it to just this list, no extra commentary.")
    try:
        return compound([{"role": "user", "content": q}])
    except Exception:
        return ""


# ------------------------------------------------- itinerary: structured JSON --
ITEM_TYPES = ("sight", "food", "transport", "stay", "activity", "shopping", "nightlife")

_JSON_SPEC = """OUTPUT FORMAT (this replaces any formatting instructions above: ignore the Markdown / table parts, but keep following EVERY other rule, including realistic prices, distances and interest coverage).
Return ONLY one valid JSON object. No prose, no markdown fences, no comments. Exactly this shape:
{
  "title": "short trip title",
  "summary": "two-sentence overview of the trip (max 40 words)",
  "days": [
    {"day": 1, "city": "the city this day is in", "theme": "3-6 word theme",
     "items": [
       {"time": "9:00 AM", "period": "morning" | "afternoon" | "evening",
        "type": "sight" | "food" | "transport" | "stay" | "activity" | "shopping" | "nightlife",
        "title": "specific place name (for transport: 'Mode: Place A to Place B')",
        "description": "max 22 words, practical tip",
        "cost": 0,
        "cost_note": "optional, short, e.g. 'entry fee; foreigners pay more'",
        "mode": "transport items only, e.g. 'Auto-rickshaw'", "distance": "transport only, e.g. '4.2 km'", "duration": "transport only, e.g. '15 min'"}
     ],
     "subtotal": 0}
  ],
  "breakdown": [{"category": "Lodging", "amount": 0}, {"category": "Food", "amount": 0}, {"category": "Transport", "amount": 0}, {"category": "Activities", "amount": 0}, {"category": "Misc/Buffer", "amount": 0}],
  "status_note": "one or two sentences: whether the plan is under, at or over the budget and, if over, concrete ways to cut costs",
  "tips": ["2-3 money-saving tips specific to {city}"],
  "coverage_note": "one sentence: which interests were covered and any that had to be skipped, and why"
}
Rules: every cost is a plain number in {currency} per person (no currency symbols, no commas, no ranges; use 0 for free). Put each transfer between places in its own "transport" item. Keep every text field short so the whole JSON fits in the response."""


def _clean_text(v, n=200) -> str:
    return re.sub(r"\s+", " ", str(v if v is not None else "")).strip()[:n]


def _num(v, default=None):
    """Tolerant number parsing: 150, '150', '₹1,200', '150-250' (-> midpoint), 'free' (-> 0)."""
    if isinstance(v, bool):
        return default
    if isinstance(v, (int, float)):
        return float(v) if math.isfinite(v) else default
    if isinstance(v, str):
        if "free" in v.lower():
            return 0.0
        nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", v.replace(",", ""))][:2]
        if nums:
            return sum(nums) / len(nums)
    return default


def _period_from_time(t) -> str:
    m = re.search(r"(\d{1,2})(?::(\d{2}))?\s*([ap])?\.?m?", str(t or "").lower())
    if not m:
        return "morning"
    h = int(m.group(1)) % 24
    if m.group(3) == "p" and h < 12:
        h += 12
    if m.group(3) == "a" and h == 12:
        h = 0
    return "morning" if h < 12 else "afternoon" if h < 17 else "evening"


def normalize_itinerary(data, currency, budget):
    """Validate / sanitise the model's JSON. Returns None if it isn't a usable itinerary."""
    if isinstance(data, list):
        data = {"days": data}
    if not isinstance(data, dict) or not isinstance(data.get("days"), list):
        return None
    days = []
    for dd in data["days"]:
        if not isinstance(dd, dict):
            continue
        items = []
        for it in (dd.get("items") or dd.get("activities") or []):
            if not isinstance(it, dict):
                continue
            title = _clean_text(it.get("title") or it.get("name"), 120)
            if not title:
                continue
            typ = _clean_text(it.get("type"), 20).lower()
            if typ not in ITEM_TYPES:
                typ = "transport" if it.get("mode") else "activity"
            period = _clean_text(it.get("period"), 12).lower()
            if period not in ("morning", "afternoon", "evening"):
                period = _period_from_time(it.get("time"))
            cost = _num(it.get("cost"))
            row = {"time": _clean_text(it.get("time"), 14), "period": period, "type": typ, "title": title,
                   "description": _clean_text(it.get("description"), 220),
                   "cost": None if cost is None else max(0, round(cost)),
                   "cost_note": _clean_text(it.get("cost_note"), 90)}
            if typ == "transport":
                row.update(mode=_clean_text(it.get("mode"), 30), distance=_clean_text(it.get("distance"), 20),
                           duration=_clean_text(it.get("duration"), 20))
            items.append(row)
        if not items:
            continue
        costs = [x["cost"] for x in items if x["cost"] is not None]
        sub = sum(costs) if costs else (_num(dd.get("subtotal"), 0) or 0)
        days.append({"day": len(days) + 1, "city": _clean_text(dd.get("city"), 40),
                     "theme": _clean_text(dd.get("theme"), 60), "items": items, "subtotal": round(sub)})
    if not days:
        return None

    breakdown = []
    for b in (data.get("breakdown") or []):
        if not isinstance(b, dict):
            continue
        cat = _clean_text(b.get("category") or b.get("name"), 30)
        amt = _num(b.get("amount") if "amount" in b else b.get("cost"))
        if cat and amt is not None and amt >= 0 and cat.lower() not in ("total", "grand total", "estimated total"):
            breakdown.append({"category": cat, "amount": round(amt)})
    if not breakdown:  # derive it from the items if the model skipped it
        names, agg = {"stay": "Lodging", "food": "Food", "transport": "Transport"}, {}
        for d in days:
            for x in d["items"]:
                if x["cost"]:
                    g = names.get(x["type"], "Activities")
                    agg[g] = agg.get(g, 0) + x["cost"]
        breakdown = [{"category": k, "amount": round(v)} for k, v in agg.items()]
    est = sum(b["amount"] for b in breakdown) or sum(d["subtotal"] for d in days)
    tips = [_clean_text(t, 220) for t in data.get("tips", []) if isinstance(t, str) and t.strip()][:4] \
        if isinstance(data.get("tips"), list) else []
    status = "under" if est <= budget * 0.97 else "at" if est <= budget * 1.03 else "over"
    return {"title": _clean_text(data.get("title"), 90), "summary": _clean_text(data.get("summary"), 320),
            "currency": currency, "days": days, "breakdown": breakdown, "estimated_total": round(est),
            "status": status, "status_note": _clean_text(data.get("status_note"), 400), "tips": tips,
            "coverage_note": _clean_text(data.get("coverage_note"), 300)}


def itinerary_to_markdown(it, meta) -> str:
    cur = it["currency"]

    def money(n):
        return f"{n:,.0f} {cur}"

    L = [f"# {it['title'] or meta['city'] + ' itinerary'}", "",
         f"**{meta['days']} days · budget {money(meta['budget'])} · interests: {', '.join(meta['interests'])}**", ""]
    if it["summary"]:
        L += [it["summary"], ""]
    for d in it["days"]:
        head = f"## Day {d['day']}"
        if d["city"]:
            head += f" — {d['city']}"
        if d["theme"]:
            head += f": {d['theme']}"
        L += [head, ""]
        for x in d["items"]:
            when = f"{x['time']} — " if x["time"] else ""
            extra = " · ".join(v for v in (x.get("mode"), x.get("distance"), x.get("duration")) if v)
            cost = "" if x["cost"] is None else (" · free" if x["cost"] == 0 else f" · {money(x['cost'])}")
            line = f"- {when}**{x['title']}**" + (f" ({extra})" if extra else "") + cost
            if x["description"]:
                line += f" — {x['description']}"
            if x["cost_note"]:
                line += f" _({x['cost_note']})_"
            L.append(line)
        L += ["", f"**Day {d['day']} subtotal: {money(d['subtotal'])}**", ""]
    L += ["## Budget breakdown", "", "| Category | Amount |", "|---|---:|"]
    L += [f"| {b['category']} | {money(b['amount'])} |" for b in it["breakdown"]]
    L += [f"| **Estimated total** | **{money(it['estimated_total'])}** |", ""]
    verdict = {"under": "under budget", "at": "right on budget", "over": "over budget"}[it["status"]]
    L.append(f"The plan comes out **{verdict}** (budget: {money(meta['budget'])}). {it['status_note']}".strip())
    L.append("")
    if it["tips"]:
        L += ["## Money-saving tips", ""] + [f"- {t}" for t in it["tips"]] + [""]
    if it["coverage_note"]:
        L += [f"_{it['coverage_note']}_", ""]
    return "\n".join(L)


def build_itinerary(d) -> dict:
    city = str(d.get("city", "")).strip()[:80]
    try:
        days = int(d.get("days"))
        budget = float(d.get("budget"))
    except (TypeError, ValueError):
        raise ValueError("Please enter a valid number of days and budget.")
    if not city: raise ValueError("Please enter a destination.")
    if not 1 <= days <= 14: raise ValueError("Trip length must be between 1 and 14 days.")
    if budget <= 0: raise ValueError("Budget must be greater than zero.")
    currency = str(d.get("currency", "USD")).upper()[:4]
    interests = [str(i).strip() for i in (d.get("interests") or []) if str(i).strip()][:8]
    if not interests: raise ValueError("Add at least one interest (e.g. food, heritage, nature).")

    highlights = fetch_highlights(city, interests)
    block = (f"Known highlights for {city} matching the traveler's interests (from a real web search - use these "
             f"as strong candidates, see rule 12 below on how to treat them):\n{highlights}\n\n") if highlights else ""
    fmt = dict(city=city, days=days, budget=(int(budget) if budget == int(budget) else budget), currency=currency,
               interests=", ".join(interests), highlights_block=block)
    base, fallback = ITINERARY_PROMPT.format(**fmt), ITINERARY_PROMPT_FALLBACK.format(**fmt)
    spec = _JSON_SPEC.replace("{currency}", currency).replace("{city}", city)
    nudge = "\n\nKeep the writing concise so the full itinerary fits within the response limit."
    meta = {"city": city, "days": days, "budget": fmt["budget"], "currency": currency, "interests": interests}

    # 1) structured plan with live web search -> 2) structured plan from model knowledge
    used_search, plan = True, None
    try:
        plan = normalize_itinerary(extract_json(compound(
            [{"role": "user", "content": base + "\n\n" + spec}], max_tokens=7000, timeout=80.0)), currency, budget)
    except Exception:
        plan = None
    if not plan:
        used_search = False
        try:
            plan = normalize_itinerary(extract_json(fast(
                [{"role": "user", "content": fallback + "\n\n" + spec}], max_tokens=8000, temperature=0.5, timeout=80.0)),
                currency, budget)
        except Exception:
            plan = None

    if plan:
        notice = ("Please double-check before you travel: this itinerary was generated with web-search-assisted AI, "
                  "which improves accuracy but can still be wrong or outdated. Verify entry fees, exact travel times "
                  "and opening hours on Google Maps or official sources, especially for longer or uphill routes."
                  if used_search else
                  "Please verify before you travel: the web-search-assisted model was temporarily unavailable, so this "
                  "itinerary comes from general AI knowledge rather than live lookups. Prices, distances and travel "
                  "times may be less accurate - double-check everything on Google Maps or official sources.")
        md = itinerary_to_markdown(plan, meta) + f"\n---\n⚠️ {notice}\n"
        return {"markdown": md, "itinerary": plan, "notice": notice, "used_search": used_search, **meta}

    # 3) last resort: the original free-text plan (the UI still renders it, just without the timeline)
    used_search, md = True, ""
    try:
        md = compound([{"role": "user", "content": base}]) or compound([{"role": "user", "content": base + nudge}])
    except Exception:
        md = ""
    if not md:
        used_search = False
        md = fast([{"role": "user", "content": fallback}], temperature=0.7) or \
             fast([{"role": "user", "content": fallback + nudge}], temperature=0.7)
    if not md:
        raise RuntimeError("The itinerary came back empty (can happen with longer trips). Try again or shorten the trip.")
    if used_search:
        md += ("\n\n---\n\u26a0\ufe0f **Please double-check before you travel**: this itinerary was generated with "
               "web-search-assisted AI, which improves accuracy but can still be wrong or outdated. Verify entry fees, "
               "exact travel times, and opening hours on Google Maps or official sources before finalizing your plan, "
               "especially for longer/uphill routes and less common attractions.")
    else:
        md += ("\n\n---\n\u26a0\ufe0f **Please verify before you travel**: the web-search-assisted model was temporarily "
               "unavailable, so this itinerary was generated from general AI knowledge rather than live lookups. "
               "Prices, distances, and travel times may be less accurate \u2014 double-check everything on Google Maps "
               "or official sources before finalizing your plan.")
    return {"markdown": md, "itinerary": None, "notice": "", "used_search": used_search, **meta}


# ---------------------------------------------------------------- community --
COMM_FILE, COMM_LOCK = "saferoam_community.json", threading.Lock()


def comm_load():
    try:
        with open(COMM_FILE, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {}


# ---------------------------------------------------------------- Flask app --
app = Flask("saferoam")


def fail(msg, code=400):
    return jsonify({"error": str(msg)}), code


def body():
    return request.get_json(force=True, silent=True) or {}


def need_place(d):
    try:
        return {"name": str(d["name"]), "label": str(d.get("label") or d["name"]), "country": str(d.get("country", "")),
                "country_code": str(d.get("country_code", "")).upper(), "lat": float(d["lat"]), "lon": float(d["lon"])}
    except Exception:
        raise ValueError("Missing or invalid city data.")


@app.get("/")
def index():
    r = Response(INDEX_HTML, mimetype="text/html")
    r.headers["Cache-Control"] = "no-store"
    return r


@app.post("/api/geocode")
def api_geocode():
    try:
        return jsonify(geocode(body().get("query", "")))
    except ValueError as e:
        return fail(e, 404)
    except Exception as e:
        return fail(f"Location lookup failed: {e}", 502)


@app.get("/api/suggest")
def api_suggest():
    try:
        return jsonify({"results": suggest_places(request.args.get("q", ""))})
    except Exception as e:
        return fail(f"Search failed: {e}", 502)


@app.get("/api/weather")
def api_weather():
    try:
        return jsonify(get_weather(float(request.args["lat"]), float(request.args["lon"])))
    except Exception as e:
        return fail(f"Weather unavailable: {e}", 502)


@app.get("/api/hospitals")
def api_hospitals():
    try:
        return jsonify({"hospitals": get_hospitals(float(request.args["lat"]), float(request.args["lon"]))})
    except Exception as e:
        return fail(f"Hospital lookup failed: {e}", 502)


def _job(fn, prep):
    try:
        arg = prep(body())
    except ValueError as e:
        return fail(e)
    return jsonify({"job": start_job(fn, arg)})


@app.post("/api/places")
def api_places(): return _job(build_places, need_place)


@app.post("/api/safety")
def api_safety(): return _job(build_safety, need_place)


@app.post("/api/chat")
def api_chat(): return _job(do_chat, lambda d: d)


@app.post("/api/itinerary")
def api_itinerary(): return _job(build_itinerary, lambda d: d)


@app.get("/api/job/<jid>")
def api_job(jid):
    j = JOBS.get(jid)
    if not j:
        return fail("Unknown or expired job.", 404)
    return jsonify({k: v for k, v in j.items() if k != "ts"})


@app.get("/api/community")
def api_comm_get():
    key = request.args.get("city", "").strip().lower()
    posts = comm_load().get(key, [])
    return jsonify({"posts": list(reversed(posts))[:50]})


@app.post("/api/community")
def api_comm_post():
    d = body()
    key = str(d.get("city", "")).strip().lower()
    text = str(d.get("text", "")).strip()
    author = (str(d.get("author", "")).strip() or "Traveler")[:30]
    cat = d.get("category") if d.get("category") in ("tip", "warning", "food", "stay") else "tip"
    if not key: return fail("City missing.")
    if not 3 <= len(text) <= 400: return fail("Please write between 3 and 400 characters.")
    post = {"id": uuid.uuid4().hex[:8], "author": author, "text": text, "category": cat, "ts": int(time.time())}
    with COMM_LOCK:
        data = comm_load()
        data.setdefault(key, []).append(post)
        data[key] = data[key][-200:]
        with open(COMM_FILE, "w", encoding="utf-8") as f:
            json.dump(data, f)
    return jsonify({"post": post})


# ======================================================================= UI ===
INDEX_HTML = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>SafeRoam — AI Tourist & Safety Dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/react/18.2.0/umd/react.production.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/react-dom/18.2.0/umd/react-dom.production.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/babel-standalone/7.23.2/babel.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/marked/12.0.2/marked.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/dompurify/3.1.6/purify.min.js"></script>
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css"/>
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/html2pdf.js/0.10.1/html2pdf.bundle.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600;700&family=DM+Sans:ital,wght@0,300;0,400;0,500;0,600;1,400&display=swap" rel="stylesheet"/>
<style>
  *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
  :root {
    --bg: #0d1117; --surface: #161b22; --surface2: #1c2330;
    --border: #30363d; --accent: #f0a500; --accent2: #e05c5c;
    --safe: #3fb950; --text: #e6edf3; --muted: #8b949e; --card-bg: #1c2330;
  }
  html, body, #root { height: 100%; margin: 0; padding: 0; overflow: hidden; }
  body { font-family: 'DM Sans', sans-serif; background: var(--bg); color: var(--text); }
  button { cursor: pointer; font-family: 'DM Sans', sans-serif; }
  button:disabled { opacity: .55; cursor: not-allowed; }
  input, select, textarea { font-family: 'DM Sans', sans-serif; }
  a { color: var(--accent); }
  ::-webkit-scrollbar { width: 4px; }
  ::-webkit-scrollbar-track { background: transparent; }
  ::-webkit-scrollbar-thumb { background: var(--border); border-radius: 2px; }

  @keyframes pulse-sos { 0%, 100% { box-shadow: 0 0 0 0 rgba(224,92,92,0.5); } 50% { box-shadow: 0 0 0 8px rgba(224,92,92,0); } }
  @keyframes blink { 0%,100%{opacity:1} 50%{opacity:0.3} }
  @keyframes fadeIn { from{opacity:0;transform:translateY(6px)} to{opacity:1;transform:translateY(0)} }
  @keyframes spin { to { transform: rotate(360deg); } }
  .spinner { width:14px; height:14px; border:2px solid var(--border); border-top-color: var(--accent); border-radius:50%; animation: spin .8s linear infinite; display:inline-block; vertical-align:middle; }

  .md { font-size:.84rem; line-height:1.55; color: var(--text); }
  .md h1, .md h2, .md h3, .md h4 { font-family:'Playfair Display',serif; color: var(--accent); margin: 16px 0 8px; }
  .md h1 { font-size:1.3rem; } .md h2 { font-size:1.12rem; } .md h3 { font-size:1rem; } .md h4 { font-size:.9rem; }
  .md p { margin: 6px 0; }
  .md ul, .md ol { margin: 6px 0 6px 22px; }
  .md li { margin: 2px 0; }
  .md strong { color: #fff; }
  .md table { border-collapse: collapse; margin: 10px 0; width: 100%; font-size:.78rem; display:block; overflow-x:auto; }
  .md th, .md td { border: 1px solid var(--border); padding: 5px 9px; text-align: left; vertical-align: top; }
  .md th { background: var(--surface2); }
  .md hr { border: none; border-top: 1px solid var(--border); margin: 14px 0; }
  .field { display:flex; flex-direction:column; gap:5px; margin-bottom:12px; }
  .field label { font-size:.72rem; text-transform:uppercase; letter-spacing:.08em; color: var(--muted); font-weight:600; }
  .inp { background: var(--bg); border:1px solid var(--border); border-radius:8px; padding:8px 12px; color: var(--text); font-size:.85rem; outline:none; width:100%; }
  .inp:focus { border-color: var(--accent); }
  .chip { display:inline-block; border-radius:20px; padding:4px 11px; font-size:.74rem; cursor:pointer; border:1px solid rgba(240,165,0,.25); background: rgba(240,165,0,.09); color: var(--accent); user-select:none; }
  .chip.on { background: var(--accent); color:#000; font-weight:600; }
  .btn { background: var(--accent); color:#000; border:none; border-radius:20px; padding:8px 20px; font-size:.82rem; font-weight:700; }
  .btn.ghost { background:none; color: var(--text); border:1px solid var(--border); font-weight:500; }
  .btn.danger { background: var(--accent2); color:#fff; }
  .sect { font-size:.72rem; text-transform:uppercase; letter-spacing:.09em; color: var(--muted); margin-bottom:7px; font-weight:600; }

  /* ───────── live map (Leaflet) ───────── */
  .leaflet-container { background:#0a1628; font-family:'DM Sans',sans-serif; outline:none; }
  .sr-pin-wrap { background:none; border:none; }
  .sr-pin { width:26px; height:26px; border-radius:50%; background:var(--c); border:2px solid rgba(255,255,255,.92);
    box-shadow:0 0 0 4px var(--g), 0 2px 8px rgba(0,0,0,.55); display:flex; align-items:center; justify-content:center;
    color:#fff; font-size:12px; font-weight:700; transition:transform .15s, box-shadow .15s; }
  .sr-pin.active { transform:scale(1.3); box-shadow:0 0 0 6px var(--g), 0 0 20px var(--c); }
  .sr-hosp { width:26px; height:26px; border-radius:8px; background:#fff; border:2px solid #e05c5c; display:flex; align-items:center; justify-content:center; font-size:14px; box-shadow:0 2px 8px rgba(0,0,0,.5); }
  .sr-user { width:18px; height:18px; border-radius:50%; background:#60a5fa; border:3px solid #fff; box-shadow:0 0 0 0 rgba(96,165,250,.6); animation:pulse-user 2s infinite; }
  @keyframes pulse-user { 0%{box-shadow:0 0 0 0 rgba(96,165,250,.55)} 70%{box-shadow:0 0 0 14px rgba(96,165,250,0)} 100%{box-shadow:0 0 0 0 rgba(96,165,250,0)} }
  .leaflet-tooltip.sr-tip { background:var(--surface2); color:var(--text); border:1px solid var(--border); box-shadow:0 4px 14px rgba(0,0,0,.45); font-size:.74rem; font-weight:600; padding:4px 9px; }
  .leaflet-tooltip-top.sr-tip::before { border-top-color:var(--border); }
  .leaflet-popup-content-wrapper, .leaflet-popup-tip { background:var(--surface2); color:var(--text); border:1px solid var(--border); box-shadow:0 8px 24px rgba(0,0,0,.5); }
  .leaflet-popup-content { margin:11px 14px; font-size:.8rem; line-height:1.45; }
  .leaflet-popup-content a { color:var(--accent); }
  .sr-pop-t { font-weight:700; font-size:.92rem; margin-bottom:2px; }
  .sr-pop-a { color:var(--muted); font-size:.74rem; margin-bottom:6px; }
  .leaflet-bar a, .leaflet-bar a:hover { background:var(--surface2); color:var(--text); border-color:var(--border); }
  .leaflet-bar a:hover { background:var(--border); }
  .leaflet-control-attribution { background:rgba(13,17,23,.78) !important; color:var(--muted) !important; font-size:10px; }
  .leaflet-control-attribution a { color:var(--accent) !important; }
  .map-btn { background:rgba(22,27,34,.92); border:1px solid var(--border); color:var(--text); border-radius:8px; padding:5px 11px; font-size:.74rem; font-weight:500; backdrop-filter:blur(6px); }
  .map-btn:hover { border-color:var(--accent); }
  .map-btn.on { background:var(--accent); color:#000; border-color:var(--accent); font-weight:700; }
  .map-seg { display:flex; border-radius:8px; overflow:hidden; border:1px solid var(--border); background:rgba(22,27,34,.92); backdrop-filter:blur(6px); }
  .map-seg button { background:none; border:none; color:var(--muted); padding:5px 11px; font-size:.74rem; font-weight:500; }
  .map-seg button.on { background:var(--accent); color:#000; font-weight:700; }

  /* ───────── destination search (typeahead) ───────── */
  .ps-wrap { position:relative; }
  .ps-box { display:flex; align-items:center; gap:8px; background:var(--bg); border:1px solid var(--border); border-radius:20px; padding:6px 13px; }
  .ps-box.sq { border-radius:8px; padding:8px 12px; }
  .ps-box:focus-within { border-color:var(--accent); }
  .ps-box input { flex:1; min-width:0; background:none; border:none; outline:none; color:var(--text); font-size:.8rem; }
  .ps-x { color:var(--muted); font-size:.74rem; cursor:pointer; padding:0 2px; }
  .ps-list { position:absolute; left:0; top:calc(100% + 6px); z-index:2000; min-width:100%; width:max-content; max-width:340px; background:var(--surface2); border:1px solid var(--border); border-radius:12px; box-shadow:0 14px 36px rgba(0,0,0,.55); overflow:hidden; }
  .ps-opt { display:flex; flex-direction:column; gap:1px; padding:8px 14px; cursor:pointer; font-size:.82rem; }
  .ps-opt small { color:var(--muted); font-size:.72rem; }
  .ps-opt.hi { background:rgba(240,165,0,.13); }
  .ps-empty { padding:10px 14px; font-size:.78rem; color:var(--muted); }

  /* ───────── trip planner form ───────── */
  .dest-tags { display:flex; flex-wrap:wrap; gap:6px; margin-bottom:7px; }
  .dest-tag { display:inline-flex; align-items:center; gap:4px; background:rgba(240,165,0,.12); border:1px solid rgba(240,165,0,.3); color:var(--accent); border-radius:16px; padding:3px 5px 3px 11px; font-size:.78rem; }
  .dest-tag button { background:none; border:none; color:inherit; font-size:.8rem; padding:0 5px; }
  .stepper { display:flex; align-items:center; border:1px solid var(--border); border-radius:8px; overflow:hidden; background:var(--bg); }
  .stepper button { background:none; border:none; color:var(--text); width:38px; height:36px; font-size:1.05rem; }
  .stepper button:hover:not(:disabled) { background:var(--surface2); }
  .stepper span { flex:1; text-align:center; font-size:.88rem; font-weight:600; }

  /* ───────── itinerary view ───────── */
  .it-head { display:flex; gap:16px; align-items:flex-start; flex-wrap:wrap; margin-bottom:14px; }
  .it-title { font-family:'Playfair Display',serif; font-size:1.75rem; line-height:1.15; color:var(--accent); }
  .it-meta { display:flex; flex-wrap:wrap; gap:6px; margin-top:10px; }
  .it-actions { display:flex; gap:8px; align-items:center; position:relative; }
  .pill { display:inline-flex; align-items:center; gap:6px; font-size:.72rem; padding:3px 10px; border-radius:12px; border:1px solid var(--border); color:var(--muted); background:var(--surface); text-decoration:none; }
  .pill.good { color:var(--safe); border-color:rgba(63,185,80,.4); background:rgba(63,185,80,.1); }
  .pill.bad { color:var(--accent2); border-color:rgba(224,92,92,.4); background:rgba(224,92,92,.1); }
  .pill.link { color:var(--accent); border-color:rgba(240,165,0,.35); }
  .pill.link:hover { background:rgba(240,165,0,.12); }
  .it-tabs { position:sticky; top:0; z-index:5; display:flex; gap:6px; overflow-x:auto; padding:10px 0 12px; margin-bottom:14px; background:var(--bg); border-bottom:1px solid var(--border); }
  .it-tab { flex:0 0 auto; text-align:left; background:var(--surface); border:1px solid var(--border); color:var(--muted); border-radius:10px; padding:7px 13px; min-width:90px; }
  .it-tab b { display:block; font-size:.8rem; color:var(--text); font-weight:600; }
  .it-tab span { display:block; font-size:.68rem; max-width:140px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; margin-top:1px; }
  .it-tab:hover { border-color:var(--accent); }
  .it-tab.on { border-color:var(--accent); background:rgba(240,165,0,.1); }
  .it-tab.on b { color:var(--accent); }
  .it-tab:focus-visible, .btn:focus-visible, .dl-item:focus-visible, .ov-day:focus-visible, .map-btn:focus-visible { outline:2px solid var(--accent); outline-offset:2px; }

  .ov-sum { font-size:.92rem; line-height:1.6; color:var(--text); max-width:68ch; margin-bottom:4px; }
  .bm { background:var(--card-bg); border:1px solid var(--border); border-radius:12px; padding:16px 18px 18px; margin:14px 0; }
  .bm-top { display:flex; align-items:baseline; gap:10px; flex-wrap:wrap; }
  .bm-amt { font-family:'Playfair Display',serif; font-size:1.55rem; color:var(--text); }
  .bm-track { position:relative; margin:22px 0 14px; height:12px; }
  .bm-segs { display:flex; height:100%; border-radius:6px; overflow:hidden; background:#262e3a; }
  .bm-segs span { display:block; height:100%; min-width:2px; }
  .bm-mark { position:absolute; top:-5px; bottom:-5px; width:2px; background:var(--text); border-radius:1px; }
  .bm-mark::after { content:"Budget"; position:absolute; top:-16px; right:6px; font-size:.64rem; color:var(--muted); font-style:normal; white-space:nowrap; }
  .bm-legend { display:grid; grid-template-columns:repeat(auto-fill,minmax(190px,1fr)); gap:8px 20px; }
  .bm-li { display:flex; align-items:center; gap:8px; font-size:.78rem; color:var(--muted); }
  .bm-li i { width:9px; height:9px; border-radius:3px; flex:0 0 auto; }
  .bm-li b { color:var(--text); font-weight:600; margin-left:auto; }
  .ov-note { font-size:.84rem; line-height:1.55; color:var(--text); margin:0 0 6px; max-width:68ch; }
  .ov-days { display:grid; grid-template-columns:repeat(auto-fill,minmax(235px,1fr)); gap:10px; margin:8px 0 20px; }
  .ov-day { text-align:left; background:var(--card-bg); border:1px solid var(--border); border-radius:12px; padding:12px 14px; color:var(--text); transition:border-color .15s; }
  .ov-day:hover { border-color:var(--accent); }
  .ov-day .t { font-weight:600; font-size:.88rem; }
  .ov-day .s { font-size:.76rem; color:var(--muted); margin-top:3px; line-height:1.4; }
  .ov-day .f { display:flex; justify-content:space-between; margin-top:9px; font-size:.74rem; color:var(--muted); }
  .ov-day .f b { color:var(--accent); font-weight:600; }
  .tip-list { margin:6px 0 0 18px; font-size:.84rem; line-height:1.6; max-width:68ch; }
  .callout { border:1px solid rgba(240,165,0,.35); background:rgba(240,165,0,.07); border-radius:10px; padding:10px 14px; font-size:.76rem; color:var(--muted); line-height:1.5; margin-top:20px; max-width:68ch; }

  .day-head { display:flex; justify-content:space-between; align-items:flex-start; gap:12px; flex-wrap:wrap; margin-bottom:16px; }
  .day-title { font-family:'Playfair Display',serif; font-size:1.3rem; color:var(--text); }
  .day-theme { font-size:.82rem; color:var(--muted); margin-top:3px; }
  .day-side { display:flex; gap:7px; flex-wrap:wrap; }
  .tl-group { margin-bottom:20px; }
  .tl-gh { font-size:.8rem; font-weight:600; color:var(--muted); margin-bottom:10px; }
  .tl { border-left:2px solid var(--border); margin-left:6px; padding-left:22px; }
  .tl-item { position:relative; margin-bottom:10px; }
  .tl-item::before { content:""; position:absolute; left:-29px; top:17px; width:12px; height:12px; border-radius:50%; background:var(--c); border:2px solid var(--bg); }
  .tl-item.move::before { top:9px; width:8px; height:8px; left:-27px; background:var(--bg); border:2px solid var(--c); }
  .tl-card { display:grid; grid-template-columns:1fr auto; gap:14px; background:var(--card-bg); border:1px solid var(--border); border-radius:12px; padding:12px 15px; }
  .tl-top { display:flex; gap:10px; align-items:center; font-size:.72rem; margin-bottom:3px; }
  .tl-time { color:var(--muted); font-weight:600; }
  .tl-title { font-size:.94rem; font-weight:600; line-height:1.35; }
  .tl-desc { font-size:.8rem; color:var(--muted); line-height:1.5; margin-top:3px; }
  .tl-map { display:inline-block; margin-top:6px; font-size:.74rem; }
  .tl-cost { text-align:right; min-width:70px; }
  .cost { font-weight:700; font-size:.86rem; color:var(--text); white-space:nowrap; }
  .cost-note { font-size:.68rem; color:var(--muted); margin-top:3px; max-width:140px; margin-left:auto; line-height:1.35; }
  .mv { display:flex; align-items:baseline; gap:10px; flex-wrap:wrap; font-size:.78rem; color:var(--muted); padding:2px 0; }
  .mv-bits { color:var(--text); }
  .mv .cost { margin-left:auto; font-size:.78rem; color:var(--muted); font-weight:600; }
  .mv-d { font-size:.74rem; color:var(--muted); margin-top:1px; }
  .day-foot { display:flex; justify-content:flex-end; align-items:baseline; gap:10px; border-top:1px solid var(--border); padding-top:12px; margin-top:6px; font-size:.82rem; color:var(--muted); }
  .day-foot b { font-size:1.05rem; color:var(--accent); font-family:'Playfair Display',serif; }

  .dl-menu { position:absolute; right:0; top:calc(100% + 6px); z-index:50; width:300px; background:var(--surface2); border:1px solid var(--border); border-radius:12px; box-shadow:0 14px 36px rgba(0,0,0,.55); padding:6px; }
  .dl-item { display:block; width:100%; text-align:left; background:none; border:none; color:var(--text); padding:9px 12px; border-radius:8px; }
  .dl-item:hover { background:rgba(240,165,0,.1); }
  .dl-item b { display:block; font-size:.84rem; font-weight:600; }
  .dl-item span { display:block; font-size:.72rem; color:var(--muted); margin-top:2px; }
  .dl-err { position:absolute; right:0; top:calc(100% + 6px); width:300px; font-size:.74rem; color:var(--accent2); background:var(--surface2); border:1px solid rgba(224,92,92,.4); border-radius:10px; padding:8px 12px; z-index:50; line-height:1.4; }

  @keyframes shimmer { to { background-position:-200% 0; } }
  .sk { background:linear-gradient(90deg,#1c2330 25%,#252f40 50%,#1c2330 75%); background-size:200% 100%; animation:shimmer 1.5s linear infinite; border-radius:10px; }
  @media (prefers-reduced-motion: reduce) { .sk, .sr-user, .spinner { animation:none; } }
  @media (max-width: 760px) { .tl-card { grid-template-columns:1fr; } .tl-cost { text-align:left; display:flex; gap:10px; align-items:baseline; } .cost-note { margin-left:0; max-width:none; } }
</style>
</head>
<body>
<div id="root"></div>

<script type="text/babel">
const { useState, useRef, useEffect, useMemo } = React;

// ── CONSTANTS & HELPERS ───────────────────────────────────────────────────────
const SAFETY_COLOR = { safe: "#3fb950", moderate: "#f0a500", caution: "#e05c5c" };
const SAFETY_SCORE = { safe: 100, moderate: 65, caution: 35 };
const HEADERS = { "Content-Type": "application/json" };
const sleep = (ms) => new Promise(r => setTimeout(r, ms));

// NOTE: relative URLs on purpose — works behind Colab's port proxy as well as on localhost.
async function getJSON(url, opts) {
  const res = await fetch(url, opts);
  const data = await res.json().catch(() => ({}));
  if (!res.ok || data.error) throw new Error(data.error || `Request failed (${res.status})`);
  return data;
}
const post = (path, body) => getJSON(path, { method: "POST", headers: HEADERS, body: JSON.stringify(body) });

async function runJob(path, body, timeoutMs = 300000) {
  const { job } = await post(path, body);
  const t0 = Date.now();
  while (Date.now() - t0 < timeoutMs) {
    await sleep(1500);
    const r = await getJSON(`api/job/${job}`);
    if (r.status === "done") return r.result;
    if (r.status === "error") throw new Error(r.error);
  }
  throw new Error("Timed out waiting for the server.");
}

const lsGet = (k, d) => { try { const v = localStorage.getItem(k); return v === null ? d : v; } catch (e) { return d; } };
const lsSet = (k, v) => { try { localStorage.setItem(k, v); } catch (e) {} };

function haversine(a, b, c, d) {
  const R = 6371, rad = Math.PI / 180;
  const dp = (c - a) * rad, dl = (d - b) * rad;
  const x = Math.sin(dp / 2) ** 2 + Math.cos(a * rad) * Math.cos(c * rad) * Math.sin(dl / 2) ** 2;
  return 2 * R * Math.asin(Math.sqrt(x));
}

function renderMarkdown(md) {
  try { return DOMPurify.sanitize(marked.parse(md || "")); } catch (e) { return DOMPurify.sanitize(String(md || "")); }
}

function copyText(text) {
  try { if (navigator.clipboard && window.isSecureContext) { navigator.clipboard.writeText(text); return true; } } catch (e) {}
  try {
    const t = document.createElement("textarea"); t.value = text; document.body.appendChild(t);
    t.select(); const ok = document.execCommand("copy"); document.body.removeChild(t); return ok;
  } catch (e) { return false; }
}

const firstNumber = (s) => { const m = String(s || "").match(/\+?\d[\d\s-]*/); return m ? m[0].replace(/[^\d+]/g, "") : ""; };

// ── SHARED HELPERS: escaping, money, file saving ──────────────────────────────
const esc = (s) => String(s == null ? "" : s).replace(/[&<>"']/g, (c) => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;" }[c]));
const slug = (s) => String(s || "trip").replace(/[^\w]+/g, "-").replace(/^-+|-+$/g, "").toLowerCase() || "trip";
const money = (n, cur) => {
  const v = Math.round(Number(n) || 0);
  try { return new Intl.NumberFormat(cur === "INR" ? "en-IN" : "en", { style: "currency", currency: cur, maximumFractionDigits: 0 }).format(v); }
  catch (e) { return `${v.toLocaleString()} ${cur}`; }
};
function saveBlob(blob, name) {
  const a = document.createElement("a");
  a.href = URL.createObjectURL(blob); a.download = name;
  document.body.appendChild(a); a.click(); a.remove();
  setTimeout(() => URL.revokeObjectURL(a.href), 4000);
}

// ── ITINERARY CONSTANTS ───────────────────────────────────────────────────────
const PERIOD_META = { morning: ["☀️", "Morning"], afternoon: ["🌤️", "Afternoon"], evening: ["🌙", "Evening"] };
const TYPE_META = {
  sight: ["🏛️", "#f0a500", "Sight"], food: ["🍽️", "#3fb950", "Food"], transport: ["🚕", "#60a5fa", "Transport"],
  stay: ["🏨", "#a78bfa", "Stay"], activity: ["🎯", "#e05c5c", "Activity"], shopping: ["🛍️", "#f472b6", "Shopping"],
  nightlife: ["🎶", "#c084fc", "Nightlife"],
};
const BUDGET_COLORS = ["#f0a500", "#3fb950", "#60a5fa", "#a78bfa", "#e05c5c", "#f472b6", "#2dd4bf"];
const STATUS_META = { under: ["Under budget", "good"], at: ["On budget", "good"], over: ["Over budget", "bad"] };
const tripStatus = (it, budget) => { const e = it.estimated_total; return e <= budget * 0.97 ? "under" : e <= budget * 1.03 ? "at" : "over"; };
const mapsSearch = (q) => `https://www.google.com/maps/search/?api=1&query=${encodeURIComponent(q)}`;
const dayRoute = (d) => {
  const stops = d.items.filter(x => x.type !== "transport" && x.type !== "stay").slice(0, 9)
    .map(x => d.city ? `${x.title}, ${d.city}` : x.title);
  return stops.length >= 2 ? "https://www.google.com/maps/dir/" + stops.map(encodeURIComponent).join("/") : null;
};

// ── ITINERARY → standalone document (used for the .html and .pdf downloads) ────
const ITIN_DOC_CSS = `
.itin-doc{font-family:'DM Sans','Segoe UI',Helvetica,Arial,sans-serif;color:#1f2933;line-height:1.5;max-width:780px;margin:0 auto;padding:26px 24px;background:#fff;font-size:13px}
.itin-doc *{box-sizing:border-box}
.itin-doc h1{font-family:'Playfair Display',Georgia,serif;font-size:28px;line-height:1.15;margin:0 0 4px;color:#7a4b00}
.itin-doc h2{font-family:'Playfair Display',Georgia,serif;font-size:19px;margin:0;color:#1f2933}
.itin-doc h3{font-size:14px;margin:22px 0 6px;color:#1f2933}
.itin-doc .d-meta{color:#5b6670;font-size:12px;margin-bottom:10px}
.itin-doc .d-pill{display:inline-block;font-size:11px;font-weight:600;padding:2px 10px;border-radius:10px;margin-left:6px}
.itin-doc .d-pill.good{background:#e3f6e8;color:#1c7a33}.itin-doc .d-pill.bad{background:#fde8e8;color:#b42323}
.itin-doc .d-summary{margin:10px 0 16px;font-size:13.5px}
.itin-doc .d-bar{position:relative;margin:14px 0 10px;height:12px}
.itin-doc .d-segs{display:flex;height:100%;border-radius:6px;overflow:hidden;background:#e8ecf0}
.itin-doc .d-segs span{display:block;height:100%}
.itin-doc .d-mark{position:absolute;top:-4px;bottom:-4px;width:2px;background:#1f2933}
.itin-doc table{width:100%;border-collapse:collapse;margin:6px 0 4px;font-size:12.5px}
.itin-doc td,.itin-doc th{padding:5px 8px;border-bottom:1px solid #e8ecf0;text-align:left;vertical-align:top}
.itin-doc td.n,.itin-doc th.n{text-align:right;white-space:nowrap}
.itin-doc .dot{display:inline-block;width:9px;height:9px;border-radius:3px;margin-right:7px}
.itin-doc .d-day{margin:26px 0 8px;padding:10px 14px;background:#fdf6e7;border-left:4px solid #f0a500;border-radius:4px;display:flex;justify-content:space-between;align-items:baseline;gap:12px}
.itin-doc .d-theme{color:#5b6670;font-size:12px;margin-top:2px}
.itin-doc .d-sub{font-weight:700;white-space:nowrap}
.itin-doc .d-period{font-weight:700;font-size:12px;color:#5b6670;margin:14px 0 6px}
.itin-doc .d-item{padding:8px 0 8px 12px;border-left:3px solid var(--c,#f0a500);margin:0 0 8px 4px}
.itin-doc .d-row{display:flex;gap:10px;align-items:baseline}
.itin-doc .d-time{color:#5b6670;font-size:11.5px;font-weight:600;min-width:62px}
.itin-doc .d-title{font-weight:700;flex:1}
.itin-doc .d-cost{font-weight:700;white-space:nowrap}
.itin-doc .d-desc{color:#4a5560;margin:2px 0 0 72px;font-size:12.5px}
.itin-doc .d-note{color:#6b7680;margin:1px 0 0 72px;font-size:11.5px}
.itin-doc .d-move{margin:0 0 8px 4px;padding:3px 0 3px 12px;border-left:3px dotted #9aa5b1;color:#5b6670;font-size:12px;display:flex;gap:10px}
.itin-doc .d-move .d-cost{margin-left:auto;font-weight:600;color:#5b6670}
.itin-doc .d-total{text-align:right;font-weight:700;margin:4px 0 0}
.itin-doc ul{margin:4px 0 0 18px;padding:0}
.itin-doc .d-note-box{margin-top:18px;padding:10px 14px;background:#fff8e6;border:1px solid #f3dfa8;border-radius:6px;font-size:12px;color:#5b4a1a}
.itin-doc .d-foot{margin-top:22px;font-size:11px;color:#8a949e;border-top:1px solid #e8ecf0;padding-top:8px}
.itin-doc .d-md h1,.itin-doc .d-md h2,.itin-doc .d-md h3{margin:16px 0 6px}.itin-doc .d-md p{margin:5px 0}.itin-doc .d-md hr{border:none;border-top:1px solid #e8ecf0;margin:14px 0}
.itin-doc .d-md table{display:table}
@media print{.itin-doc{padding:0;max-width:none}.itin-doc .d-item,.itin-doc .d-move,.itin-doc .d-day{break-inside:avoid;page-break-inside:avoid}}
`;

function itineraryBodyHTML(trip) {
  const it = trip.itinerary, cur = trip.currency;
  const meta = `${trip.days} days · budget ${money(trip.budget, cur)} · ${(trip.interests || []).join(", ")}`;
  const foot = `<div class="d-foot">Generated by SafeRoam on ${esc(new Date().toLocaleDateString())}.</div>`;
  if (!it) {
    return `<h1>${esc(trip.city)}</h1><div class="d-meta">${esc(meta)}</div><div class="d-md">${renderMarkdown(trip.markdown)}</div>${foot}`;
  }
  const st = tripStatus(it, trip.budget), est = it.estimated_total, scale = Math.max(est, trip.budget) || 1;
  const stText = STATUS_META[st][0] + (st === "over" ? ` by ${money(est - trip.budget, cur)}` : st === "under" ? ` · ${money(trip.budget - est, cur)} left` : "");
  const segs = it.breakdown.map((b, i) => `<span style="width:${(b.amount / scale * 100).toFixed(2)}%;background:${BUDGET_COLORS[i % BUDGET_COLORS.length]}"></span>`).join("");
  const rows = it.breakdown.map((b, i) => `<tr><td><span class="dot" style="background:${BUDGET_COLORS[i % BUDGET_COLORS.length]}"></span>${esc(b.category)}</td><td class="n">${money(b.amount, cur)}</td><td class="n">${est ? Math.round(b.amount / est * 100) : 0}%</td></tr>`).join("");
  const days = it.days.map(d => {
    const groups = ["morning", "afternoon", "evening"].map(p => {
      const items = d.items.filter(x => x.period === p);
      if (!items.length) return "";
      return `<div class="d-period">${PERIOD_META[p][0]} ${PERIOD_META[p][1]}</div>` + items.map(x => {
        const col = (TYPE_META[x.type] || TYPE_META.activity)[1], icon = (TYPE_META[x.type] || TYPE_META.activity)[0];
        const c = x.cost == null ? "" : x.cost === 0 ? "Free" : money(x.cost, cur);
        if (x.type === "transport") {
          const bits = [x.mode && !x.title.toLowerCase().includes(x.mode.toLowerCase()) ? x.mode : "", x.distance, x.duration].filter(Boolean).join(" · ");
          return `<div class="d-move">${icon} <span>${esc(x.title)}${bits ? " · " + esc(bits) : ""}</span><span class="d-cost">${c}</span></div>`;
        }
        return `<div class="d-item" style="--c:${col}"><div class="d-row"><span class="d-time">${esc(x.time)}</span><span class="d-title">${icon} ${esc(x.title)}</span><span class="d-cost">${c}</span></div>`
          + (x.description ? `<div class="d-desc">${esc(x.description)}</div>` : "") + (x.cost_note ? `<div class="d-note">${esc(x.cost_note)}</div>` : "") + `</div>`;
      }).join("");
    }).join("");
    return `<div class="d-day"><div><h2>Day ${d.day}${d.city ? " · " + esc(d.city) : ""}</h2>${d.theme ? `<div class="d-theme">${esc(d.theme)}</div>` : ""}</div><div class="d-sub">${money(d.subtotal, cur)}</div></div>${groups}`;
  }).join("");
  return `<h1>${esc(it.title || trip.city)}</h1><div class="d-meta">${esc(trip.city)} · ${esc(meta)}</div>`
    + (it.summary ? `<div class="d-summary">${esc(it.summary)}</div>` : "")
    + `<h3>Budget <span class="d-pill ${STATUS_META[st][1]}">${esc(stText)}</span></h3>`
    + `<div class="d-bar"><div class="d-segs">${segs}</div><div class="d-mark" style="left:${(trip.budget / scale * 100).toFixed(2)}%"></div></div>`
    + `<table><tr><th>Category</th><th class="n">Amount</th><th class="n">Share</th></tr>${rows}<tr><td><b>Estimated total</b></td><td class="n"><b>${money(est, cur)}</b></td><td class="n">of ${money(trip.budget, cur)}</td></tr></table>`
    + (it.status_note ? `<p>${esc(it.status_note)}</p>` : "")
    + days
    + (it.tips && it.tips.length ? `<h3>Money-saving tips</h3><ul>${it.tips.map(t => `<li>${esc(t)}</li>`).join("")}</ul>` : "")
    + (it.coverage_note ? `<p style="color:#5b6670;font-style:italic">${esc(it.coverage_note)}</p>` : "")
    + (trip.notice ? `<div class="d-note-box">⚠️ ${esc(trip.notice)}</div>` : "") + foot;
}

function itineraryDocument(trip) {
  const title = esc((trip.itinerary && trip.itinerary.title) || trip.city);
  return `<!DOCTYPE html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>${title} · SafeRoam itinerary</title>`
    + `<link href="https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600;700&family=DM+Sans:wght@400;500;600;700&display=swap" rel="stylesheet">`
    + `<style>body{margin:0;background:#f3f4f6}${ITIN_DOC_CSS}</style></head><body><div class="itin-doc">${itineraryBodyHTML(trip)}</div></body></html>`;
}

function printTrip(trip) {
  const w = window.open("", "_blank");
  if (!w) throw new Error("Your browser blocked the pop-up. Allow pop-ups for this page, or use the HTML download.");
  w.document.open(); w.document.write(itineraryDocument(trip)); w.document.close();
  setTimeout(() => { try { w.focus(); w.print(); } catch (e) {} }, 600);
}

async function downloadPDF(trip) {
  if (!window.html2pdf) return printTrip(trip);   // PDF library didn't load -> browser print dialog ("Save as PDF")
  const holder = document.createElement("div");
  holder.innerHTML = `<style>${ITIN_DOC_CSS}</style><div class="itin-doc" style="width:780px">${itineraryBodyHTML(trip)}</div>`;
  await window.html2pdf().set({
    margin: [10, 8, 12, 8], filename: `${slug(trip.city)}-itinerary.pdf`,
    image: { type: "jpeg", quality: 0.95 },
    html2canvas: { scale: 2, useCORS: true, backgroundColor: "#ffffff", windowWidth: 800 },
    jsPDF: { unit: "mm", format: "a4", orientation: "portrait" },
    pagebreak: { mode: ["css", "legacy"], avoid: [".d-item", ".d-move", ".d-day", "tr"] },
  }).from(holder).save();
}

// ── PLACE SEARCH: live typeahead against /api/suggest ─────────────────────────
const PlaceSearch = ({ value, placeholder, onPick, onSubmitText, clearOnPick, selectAllOnFocus, square }) => {
  const [q, setQ] = useState(value || "");
  const [items, setItems] = useState([]);
  const [open, setOpen] = useState(false);
  const [busy, setBusy] = useState(false);
  const [hi, setHi] = useState(0);
  const seq = useRef(0), programmatic = useRef(null), wrap = useRef(null);

  // keep the box in sync with an externally controlled value (e.g. the current city)
  useEffect(() => { if (value !== undefined) { programmatic.current = value || ""; setQ(value || ""); } }, [value]);

  useEffect(() => {
    const text = q.trim();
    if ((programmatic.current !== null && q === programmatic.current) || text.length < 2) { seq.current++; setItems([]); setBusy(false); return; }
    const my = ++seq.current;
    setBusy(true);
    const t = setTimeout(async () => {
      try {
        const r = await getJSON(`api/suggest?q=${encodeURIComponent(text)}`);
        if (my !== seq.current) return;
        setItems(r.results || []); setHi(0);
      } catch (e) { if (my === seq.current) setItems([]); }
      if (my === seq.current) setBusy(false);
    }, 260);
    return () => clearTimeout(t);
  }, [q]);

  useEffect(() => {
    const h = (e) => { if (wrap.current && !wrap.current.contains(e.target)) setOpen(false); };
    document.addEventListener("mousedown", h);
    return () => document.removeEventListener("mousedown", h);
  }, []);

  const choose = (s) => {
    const text = clearOnPick ? "" : s.label;
    programmatic.current = text; setQ(text); setOpen(false); setItems([]);
    if (onPick) onPick(s);
  };
  const onKey = (e) => {
    if (e.key === "ArrowDown") { e.preventDefault(); setOpen(true); setHi(h => Math.min(h + 1, Math.max(items.length - 1, 0))); }
    else if (e.key === "ArrowUp") { e.preventDefault(); setHi(h => Math.max(h - 1, 0)); }
    else if (e.key === "Enter") {
      e.preventDefault();
      const t = q.trim();
      if (open && items[hi]) choose(items[hi]);
      else if (t) { setOpen(false); if (clearOnPick) { programmatic.current = ""; setQ(""); } if (onSubmitText) onSubmitText(t); }
    } else if (e.key === "Escape") { setOpen(false); }
  };

  return (
    <div className="ps-wrap" ref={wrap}>
      <div className={"ps-box" + (square ? " sq" : "")}>
        <span style={{ color: "var(--muted)", fontSize: ".8rem" }}>🔍</span>
        <input role="combobox" aria-expanded={open && items.length > 0} aria-autocomplete="list" value={q} placeholder={placeholder}
          onChange={e => { programmatic.current = null; setQ(e.target.value); setOpen(true); }}
          onKeyDown={onKey}
          onFocus={e => { if (selectAllOnFocus) e.target.select(); if (items.length) setOpen(true); }}
          onBlur={() => { if (value !== undefined) setTimeout(() => { programmatic.current = value || ""; setQ(value || ""); setOpen(false); }, 150); }}/>
        {busy ? <span className="spinner"/> : (q && !clearOnPick && value === undefined) ? <span className="ps-x" onClick={() => { programmatic.current = null; setQ(""); setItems([]); }}>✕</span> : null}
      </div>
      {open && q.trim().length >= 2 && (items.length > 0 || !busy) && (
        <div className="ps-list" role="listbox">
          {items.length === 0
            ? <div className="ps-empty">No matches yet. Press Enter to search “{q.trim()}”.</div>
            : items.map((s, i) => (
              <div key={i} role="option" aria-selected={i === hi} className={"ps-opt" + (i === hi ? " hi" : "")}
                onMouseDown={e => e.preventDefault()} onClick={() => choose(s)} onMouseEnter={() => setHi(i)}>
                <span>📍 {s.name}</span><small>{s.sub || s.country}</small>
              </div>))}
        </div>
      )}
    </div>
  );
};

// ── SUBCOMPONENTS ─────────────────────────────────────────────────────────────
const SafetyBadge = ({ safety }) => {
  const labels = { safe: "✅ Safe", moderate: "⚡ Moderate", caution: "⚠️ Caution" };
  const bgs    = { safe: "rgba(63,185,80,0.15)", moderate: "rgba(240,165,0,0.15)", caution: "rgba(224,92,92,0.15)" };
  return (
    <span style={{ fontSize:"0.7rem", padding:"2px 9px", borderRadius:10, fontWeight:600, background: bgs[safety], color: SAFETY_COLOR[safety] }}>{labels[safety]}</span>
  );
};

const DestCard = ({ dest, active, onClick }) => (
  <div onClick={onClick} style={{
    background: "var(--card-bg)", border: `1px solid ${active ? "var(--accent)" : "var(--border)"}`,
    borderRadius: 10, padding: "12px 12px 10px", marginBottom: 8, cursor: "pointer", transition: "all 0.18s",
    borderLeft: `3px solid ${active ? "var(--accent)" : "var(--border)"}`, animation: "fadeIn 0.3s ease both",
  }}>
    <div style={{fontWeight:600, fontSize:"0.9rem", marginBottom:5}}>{dest.name}</div>
    <div style={{display:"flex", alignItems:"center", gap:8, marginBottom:5}}>
      <SafetyBadge safety={dest.safety} />
      <span style={{fontSize:"0.75rem", color:"var(--accent)", marginLeft:"auto"}}>{"★".repeat(Math.round(dest.rating))} {dest.rating}</span>
    </div>
    <div style={{display:"flex", flexWrap:"wrap", gap:3, marginBottom:4}}>
      {dest.tags.map(t => (<span key={t} style={{fontSize:"0.68rem", color:"var(--muted)", background:"#21262d", padding:"2px 7px", borderRadius:8}}>{t}</span>))}
    </div>
    <div style={{fontSize:"0.72rem", color:"var(--muted)"}}>📍 {dest.area || "—"}</div>
  </div>
);

// ── LIVE MAP (Leaflet + OpenStreetMap) ────────────────────────────────────────
const TILES = {
  dark:   { label: "Dark",      url: "https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png", opt: { subdomains: "abcd", maxZoom: 19, attribution: "&copy; OpenStreetMap contributors &copy; CARTO" } },
  street: { label: "Street",    url: "https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png", opt: { subdomains: "abc", maxZoom: 19, attribution: "&copy; OpenStreetMap contributors" } },
  sat:    { label: "Satellite", url: "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}", opt: { maxZoom: 18, attribution: "Tiles &copy; Esri, Maxar, Earthstar Geographics" } },
};
const SAFETY_LABEL = { safe: "✅ Safe", moderate: "⚡ Moderate", caution: "⚠️ Caution" };

const pinIcon = (safety, active) => L.divIcon({
  className: "sr-pin-wrap",
  html: `<div class="sr-pin${active ? " active" : ""}" style="--c:${SAFETY_COLOR[safety]};--g:${SAFETY_COLOR[safety]}47"><span>${safety === "safe" ? "✓" : safety === "moderate" ? "★" : "!"}</span></div>`,
  iconSize: [30, 30], iconAnchor: [15, 15],
});

const popupHTML = (p) =>
  `<div><div class="sr-pop-t">${esc(p.name)}</div>`
  + `<div style="color:${SAFETY_COLOR[p.safety]};font-weight:600;font-size:.76rem">${SAFETY_LABEL[p.safety]} · ★ ${esc(p.rating)}</div>`
  + `<div class="sr-pop-a">${esc(p.area || "")}</div>`
  + `<a href="https://www.google.com/maps/dir/?api=1&destination=${p.lat},${p.lon}" target="_blank" rel="noopener noreferrer">Directions</a> · `
  + `<a href="https://www.google.com/maps/search/?api=1&query=${p.lat},${p.lon}" target="_blank" rel="noopener noreferrer">Open in Google Maps</a></div>`;

const MapArea = ({ place, places, visible, selected, onSelect, user, hospitals, onLocate, loading, error }) => {
  const noLeaflet = typeof L === "undefined";
  const elRef = useRef(null), mapRef = useRef(null), baseRef = useRef(null);
  const placesLayer = useRef(null), hospLayer = useRef(null), userMarker = useRef(null);
  const markers = useRef({}), byId = useRef({}), fitKey = useRef(""), wantFly = useRef(false);
  const selRef = useRef(null), onSelectRef = useRef(onSelect);
  selRef.current = selected; onSelectRef.current = onSelect;
  const [base, setBase] = useState("dark");
  const [showHosp, setShowHosp] = useState(true);

  // 1. create the map once
  useEffect(() => {
    if (noLeaflet || !elRef.current) return;
    const map = L.map(elRef.current, { zoomControl: false, worldCopyJump: true }).setView([22, 79], 4);
    L.control.zoom({ position: "bottomright" }).addTo(map);
    placesLayer.current = L.layerGroup().addTo(map);
    hospLayer.current = L.layerGroup().addTo(map);
    mapRef.current = map;
    const ro = new ResizeObserver(() => map.invalidateSize());
    ro.observe(elRef.current);
    return () => { ro.disconnect(); map.remove(); mapRef.current = null; };
  }, []);

  // 2. basemap
  useEffect(() => {
    const map = mapRef.current; if (!map) return;
    if (baseRef.current) map.removeLayer(baseRef.current);
    baseRef.current = L.tileLayer(TILES[base].url, TILES[base].opt).addTo(map);
  }, [base]);

  // 3. a new city was chosen -> fly there straight away (pins follow when the AI list arrives)
  useEffect(() => {
    const map = mapRef.current; if (!map || !place) return;
    fitKey.current = "";
    map.flyTo([place.lat, place.lon], 12, { duration: 1.2 });
  }, [place]);

  // 4. attraction pins (re-drawn when the visible list changes, e.g. while filtering)
  useEffect(() => {
    const map = mapRef.current; if (!map) return;
    placesLayer.current.clearLayers(); markers.current = {}; byId.current = {};
    visible.forEach(p => {
      byId.current[p.id] = p;
      L.circle([p.lat, p.lon], { radius: 380, stroke: false, fillColor: SAFETY_COLOR[p.safety], fillOpacity: 0.13, interactive: false }).addTo(placesLayer.current);
      const m = L.marker([p.lat, p.lon], { icon: pinIcon(p.safety, !!selRef.current && selRef.current.id === p.id), title: p.name, riseOnHover: true });
      m.bindTooltip(esc(p.name), { direction: "top", offset: [0, -14], className: "sr-tip" });
      m.bindPopup(popupHTML(p), { closeButton: false, offset: [0, -6] });
      m.on("click", () => onSelectRef.current(p));
      m.addTo(placesLayer.current);
      markers.current[p.id] = m;
    });
    const key = place ? `${place.label}|${places.length}` : "";
    if (places.length && fitKey.current !== key) {   // fit once per city, not on every keystroke in the filter box
      fitKey.current = key;
      map.fitBounds(L.latLngBounds(places.map(p => [p.lat, p.lon])), { padding: [70, 70], maxZoom: 15 });
    }
  }, [visible]);

  // 5. selection: highlight the pin, fly to it, open its card
  useEffect(() => {
    const map = mapRef.current; if (!map) return;
    Object.keys(markers.current).forEach(id => {
      const p = byId.current[id];
      markers.current[id].setIcon(pinIcon(p.safety, !!selected && String(selected.id) === String(id)));
    });
    if (!selected) { map.closePopup(); return; }
    const m = markers.current[selected.id]; if (!m) return;
    map.flyTo([selected.lat, selected.lon], Math.max(map.getZoom(), 14), { duration: 0.7 });
    const t = setTimeout(() => { if (map.hasLayer(m)) m.openPopup(); }, 760);
    return () => clearTimeout(t);
  }, [selected]);

  // 6. hospitals
  useEffect(() => {
    const map = mapRef.current; if (!map) return;
    hospLayer.current.clearLayers();
    if (!showHosp) return;
    (hospitals || []).forEach(h => {
      L.marker([h.lat, h.lon], { icon: L.divIcon({ className: "sr-pin-wrap", html: '<div class="sr-hosp">🏥</div>', iconSize: [26, 26], iconAnchor: [13, 13] }), title: h.name })
        .bindTooltip(`${esc(h.name)} · ${esc(h.dist)} km`, { direction: "top", offset: [0, -12], className: "sr-tip" })
        .addTo(hospLayer.current);
    });
  }, [hospitals, showHosp]);

  // 7. "you are here"
  useEffect(() => {
    const map = mapRef.current; if (!map) return;
    if (userMarker.current) { map.removeLayer(userMarker.current); userMarker.current = null; }
    if (!user) return;
    userMarker.current = L.marker([user.lat, user.lon], { icon: L.divIcon({ className: "sr-pin-wrap", html: '<div class="sr-user"></div>', iconSize: [20, 20], iconAnchor: [10, 10] }), zIndexOffset: 1000, title: "You are here" })
      .bindTooltip("You are here", { direction: "top", offset: [0, -10], className: "sr-tip" }).addTo(map);
    if (wantFly.current) { wantFly.current = false; map.flyTo([user.lat, user.lon], 15, { duration: 1 }); }
  }, [user]);

  const locate = () => {
    if (user && mapRef.current) mapRef.current.flyTo([user.lat, user.lon], 15, { duration: 1 });
    else { wantFly.current = true; onLocate(); }
  };

  return (
    <div style={{ flex: 1, position: "relative", background: "#0a1628", overflow: "hidden", minHeight: 120, isolation: "isolate" }}>
      <div ref={elRef} style={{ position: "absolute", inset: 0 }}/>
      {noLeaflet && (
        <div style={{ position: "absolute", inset: 0, display: "flex", alignItems: "center", justifyContent: "center", color: "var(--muted)", fontSize: ".85rem", padding: 24, textAlign: "center" }}>
          The map library couldn't load (check your internet connection and refresh).
        </div>
      )}

      {(loading || error || !places.length) && !noLeaflet && (
        <div style={{ position: "absolute", top: 14, left: "50%", transform: "translateX(-50%)", zIndex: 1100, pointerEvents: "none" }}>
          <div style={{ background: "rgba(22,27,34,0.94)", border: "1px solid var(--border)", borderRadius: 10, padding: "8px 16px", fontSize: ".8rem", color: error ? "var(--accent2)" : "var(--muted)", maxWidth: 360, textAlign: "center" }}>
            {loading ? (<><span className="spinner"/>&nbsp; Finding attractions & safety levels…</>) : (error || "No places yet.")}
          </div>
        </div>
      )}

      <div style={{ position: "absolute", top: 14, left: 14, display: "flex", flexDirection: "column", gap: 6, zIndex: 1000 }}>
        {[["safe", "Safe Zone"], ["moderate", "Moderate"], ["caution", "Caution"]].map(([k, l]) => (
          <div key={k} style={{ background: "rgba(22,27,34,0.9)", border: "1px solid var(--border)", borderRadius: 8, padding: "5px 11px", fontSize: ".74rem", display: "flex", alignItems: "center", gap: 8, backdropFilter: "blur(6px)" }}>
            <span style={{ width: 8, height: 8, borderRadius: "50%", background: SAFETY_COLOR[k], boxShadow: `0 0 6px ${SAFETY_COLOR[k]}`, display: "inline-block" }}/>{l}
          </div>
        ))}
      </div>

      <div style={{ position: "absolute", top: 14, right: 14, display: "flex", flexDirection: "column", alignItems: "flex-end", gap: 6, zIndex: 1000 }}>
        <div className="map-seg" role="group" aria-label="Map style">
          {Object.entries(TILES).map(([k, t]) => <button key={k} className={base === k ? "on" : ""} onClick={() => setBase(k)}>{t.label}</button>)}
        </div>
        <button className={"map-btn" + (showHosp ? " on" : "")} onClick={() => setShowHosp(s => !s)} aria-pressed={showHosp}>🏥 Hospitals</button>
        <button className="map-btn" onClick={locate}>📍 My location</button>
      </div>

      <div style={{ position: "absolute", bottom: 8, left: 12, zIndex: 1000, background: "rgba(22,27,34,0.88)", border: "1px solid var(--border)", borderRadius: 8, padding: "4px 10px", fontSize: ".72rem", color: "var(--muted)", backdropFilter: "blur(6px)", maxWidth: "55%" }}>
        🗺️ {place ? place.label : "…"} · safety levels are AI estimates
      </div>
    </div>
  );
};

const ChatPanel = ({ place, selected, ctx, pendingAsk, onAskDone }) => {
  const [messages, setMessages] = useState([
    { role:"ai", text:"Hello! I'm your AI travel guide. Ask me about destinations, safety tips, food, or emergencies! 🌍" }
  ]);
  const [input, setInput] = useState("");
  const [busy, setBusy] = useState(false);
  const bottomRef = useRef(null);
  const firstCity = useRef(true);

  const addAI = (text) => setMessages(m => [...m, { role:"ai", text }]);

  useEffect(() => {
    if (!selected) return;
    const s = selected.safety;
    const line = s === "safe" ? "rated safe ✅" : s === "moderate" ? "moderately safe ⚡ — stay aware" : "a busy area ⚠️ — watch your belongings";
    addAI(`Great choice! **${selected.name}** is ${line} (AI estimate). Want food spots or route guidance nearby?`);
  }, [selected?.id]);

  useEffect(() => {
    if (!place) return;
    if (firstCity.current) { firstCity.current = false; return; }
    addAI(`Switched to **${place.label}**. Ask me anything about it!`);
  }, [place?.label]);

  useEffect(() => { bottomRef.current?.scrollIntoView({ behavior:"smooth" }); }, [messages, busy]);

  const send = async (override) => {
    const t = (override ?? input).trim();
    if (!t || busy) return;
    const next = [...messages, { role:"user", text: t }];
    setMessages(next); setInput(""); setBusy(true);
    const history = next.slice(-8).map(m => ({ role: m.role === "ai" ? "assistant" : "user", content: m.text }));
    while (history.length && history[0].role === "assistant") history.shift();
    try {
      const r = await runJob("api/chat", { messages: history, context: ctx });
      addAI(r.reply);
    } catch (e) {
      addAI(`Sorry, I couldn't reach the AI service (${e.message}). Please try again.`);
    }
    setBusy(false);
  };

  useEffect(() => { if (pendingAsk) { send(pendingAsk.text); onAskDone(); } }, [pendingAsk]);

  return (
    <div style={{ height:210, borderTop:"1px solid var(--border)", display:"flex", flexDirection:"column", background:"var(--surface)", flexShrink:0 }}>
      <div style={{ padding:"9px 16px", borderBottom:"1px solid var(--border)", display:"flex", alignItems:"center", gap:8, fontSize:"0.82rem", fontWeight:600, color:"var(--accent)" }}>
        <span style={{ width:8,height:8,borderRadius:"50%",background:"var(--safe)", display:"inline-block", animation:"blink 1.5s infinite" }}/>
        AI Travel Assistant
        <span style={{color:"var(--muted)", fontWeight:400, fontSize:"0.73rem", marginLeft:"auto"}}>Powered by SafeRoam AI · live web search</span>
      </div>
      <div style={{flex:1, overflowY:"auto", padding:"10px 16px", display:"flex", flexDirection:"column", gap:7}}>
        {messages.map((m, i) => (
          <div key={i} style={{ display:"flex", gap:8, flexDirection: m.role === "user" ? "row-reverse" : "row", alignItems:"flex-start", animation:"fadeIn 0.25s ease" }}>
            <div style={{ width:24, height:24, borderRadius:"50%", display:"flex", alignItems:"center", justifyContent:"center", fontSize:"0.7rem", flexShrink:0,
              background: m.role === "ai" ? "linear-gradient(135deg,#f0a500,#e05c5c)" : "var(--border)" }}>
              {m.role === "ai" ? "🤖" : "👤"}
            </div>
            <div style={{ maxWidth:"74%", padding:"8px 12px", borderRadius:10, fontSize:"0.79rem", lineHeight:1.45, whiteSpace:"pre-wrap",
              background: m.role === "ai" ? "var(--surface2)" : "var(--accent)",
              border: m.role === "ai" ? "1px solid var(--border)" : "none",
              color: m.role === "user" ? "#000" : "var(--text)", fontWeight: m.role === "user" ? 500 : 400 }}>
              {m.text.split("**").map((p, j) => j % 2 === 1 ? <strong key={j}>{p}</strong> : <span key={j}>{p}</span>)}
            </div>
          </div>
        ))}
        {busy && (<div style={{fontSize:"0.76rem", color:"var(--muted)", paddingLeft:32}}><span className="spinner"/>&nbsp; Thinking…</div>)}
        <div ref={bottomRef}/>
      </div>
      <div style={{display:"flex", gap:8, padding:"9px 16px", borderTop:"1px solid var(--border)"}}>
        <input value={input} onChange={e => setInput(e.target.value)} onKeyDown={e => e.key === "Enter" && send()}
          placeholder="Ask about safety, food, route, emergency..."
          style={{ flex:1, background:"var(--bg)", border:"1px solid var(--border)", borderRadius:20, padding:"7px 14px", color:"var(--text)", fontSize:"0.81rem", outline:"none" }}/>
        <button onClick={() => send()} disabled={busy} style={{ background:"var(--accent)", color:"#000", border:"none", borderRadius:20, padding:"7px 18px", fontSize:"0.81rem", fontWeight:700 }}>Send</button>
      </div>
    </div>
  );
};

// ── MY TRIPS: budget itinerary planner ────────────────────────────────────────
const INTEREST_CHIPS = ["food","heritage","nature","nightlife","adventure","shopping","museums","temples","photography"];
const CURRENCIES = ["INR","USD","EUR","GBP","AED","JPY","AUD"];
const CUR_BY_COUNTRY = { IN:"INR", GB:"GBP", AE:"AED", JP:"JPY", AU:"AUD", US:"USD", DE:"EUR", FR:"EUR", ES:"EUR", IT:"EUR", NL:"EUR", PT:"EUR", GR:"EUR", IE:"EUR", AT:"EUR", BE:"EUR" };

const BudgetMeter = ({ it, budget, cur }) => {
  const est = it.estimated_total, scale = Math.max(est, budget) || 1, st = tripStatus(it, budget);
  const tail = st === "over" ? ` by ${money(est - budget, cur)}` : st === "under" ? ` · ${money(budget - est, cur)} left` : "";
  return (
    <div className="bm">
      <div className="bm-top">
        <span className="bm-amt">{money(est, cur)}</span>
        <span style={{ fontSize: ".8rem", color: "var(--muted)" }}>estimated, from a {money(budget, cur)} budget</span>
        <span className={"pill " + STATUS_META[st][1]} style={{ marginLeft: "auto" }}>{STATUS_META[st][0]}{tail}</span>
      </div>
      <div className="bm-track">
        <div className="bm-segs">
          {it.breakdown.map((b, i) => <span key={i} title={`${b.category}: ${money(b.amount, cur)}`} style={{ width: `${b.amount / scale * 100}%`, background: BUDGET_COLORS[i % BUDGET_COLORS.length] }}/>)}
        </div>
        <i className="bm-mark" style={{ left: `${budget / scale * 100}%` }}/>
      </div>
      <div className="bm-legend">
        {it.breakdown.map((b, i) => (
          <div className="bm-li" key={i}>
            <i style={{ background: BUDGET_COLORS[i % BUDGET_COLORS.length] }}/>{b.category}
            <b>{money(b.amount, cur)}</b><span style={{ width: 34, textAlign: "right" }}>{est ? Math.round(b.amount / est * 100) : 0}%</span>
          </div>
        ))}
      </div>
    </div>
  );
};

const TimelineItem = ({ x, d, cur }) => {
  const [icon, col, label] = TYPE_META[x.type] || TYPE_META.activity;
  const cost = x.cost == null ? null : x.cost === 0 ? "Free" : money(x.cost, cur);
  if (x.type === "transport") {
    const bits = [x.mode && !x.title.toLowerCase().includes(x.mode.toLowerCase()) ? x.mode : "", x.distance, x.duration].filter(Boolean).join(" · ");
    return (
      <div className="tl-item move" style={{ "--c": col }}>
        <div className="mv"><span>{icon} {x.title}</span>{bits && <span className="mv-bits">{bits}</span>}{cost && <span className="cost">{cost}</span>}</div>
        {x.description && <div className="mv-d">{x.description}</div>}
      </div>
    );
  }
  return (
    <div className="tl-item" style={{ "--c": col }}>
      <div className="tl-card">
        <div>
          <div className="tl-top">{x.time && <span className="tl-time">{x.time}</span>}<span style={{ color: col, fontWeight: 600 }}>{icon} {label}</span></div>
          <div className="tl-title">{x.title}</div>
          {x.description && <div className="tl-desc">{x.description}</div>}
          {x.type !== "stay" && <a className="tl-map" href={mapsSearch(d.city ? `${x.title}, ${d.city}` : x.title)} target="_blank" rel="noopener noreferrer">View on map ↗</a>}
        </div>
        <div className="tl-cost">
          {cost && <div className="cost">{cost}</div>}
          {x.cost_note && <div className="cost-note">{x.cost_note}</div>}
        </div>
      </div>
    </div>
  );
};

const DayView = ({ d, cur }) => {
  const route = dayRoute(d);
  return (
    <div>
      <div className="day-head">
        <div>
          <div className="day-title">Day {d.day}{d.city ? ` · ${d.city}` : ""}</div>
          {d.theme && <div className="day-theme">{d.theme}</div>}
        </div>
        <div className="day-side">
          {route && <a className="pill link" href={route} target="_blank" rel="noopener noreferrer">🗺️ Open day route in Maps</a>}
        </div>
      </div>
      {["morning", "afternoon", "evening"].map(p => {
        const items = d.items.filter(x => x.period === p);
        if (!items.length) return null;
        return (
          <div className="tl-group" key={p}>
            <div className="tl-gh">{PERIOD_META[p][0]} {PERIOD_META[p][1]}</div>
            <div className="tl">{items.map((x, i) => <TimelineItem key={i} x={x} d={d} cur={cur}/>)}</div>
          </div>
        );
      })}
      <div className="day-foot">Day {d.day} total <b>{money(d.subtotal, cur)}</b></div>
    </div>
  );
};

const Overview = ({ trip, it, onOpen }) => {
  const cur = trip.currency;
  return (
    <div>
      {it.summary && <p className="ov-sum">{it.summary}</p>}
      <BudgetMeter it={it} budget={trip.budget} cur={cur}/>
      {it.status_note && <p className="ov-note">{it.status_note}</p>}
      <div className="sect" style={{ marginTop: 18 }}>Your days</div>
      <div className="ov-days">
        {it.days.map(d => (
          <button className="ov-day" key={d.day} onClick={() => onOpen(d.day)}>
            <div className="t">Day {d.day}{d.city ? ` · ${d.city}` : ""}</div>
            <div className="s">{d.theme || d.items.filter(x => x.type !== "transport").slice(0, 3).map(x => x.title).join(", ")}</div>
            <div className="f"><span>{d.items.filter(x => x.type !== "transport" && x.type !== "stay").length} stops</span><b>{money(d.subtotal, cur)}</b></div>
          </button>
        ))}
      </div>
      {it.tips && it.tips.length > 0 && (<><div className="sect">Money-saving tips</div><ul className="tip-list">{it.tips.map((t, i) => <li key={i}>{t}</li>)}</ul></>)}
      {it.coverage_note && <p style={{ fontSize: ".78rem", color: "var(--muted)", fontStyle: "italic", marginTop: 14, maxWidth: "68ch" }}>{it.coverage_note}</p>}
    </div>
  );
};

const DownloadMenu = ({ trip }) => {
  const [open, setOpen] = useState(false);
  const [busy, setBusy] = useState(false);
  const [msg, setMsg] = useState("");
  const ref = useRef(null);
  useEffect(() => {
    const h = (e) => { if (ref.current && !ref.current.contains(e.target)) setOpen(false); };
    const k = (e) => { if (e.key === "Escape") setOpen(false); };
    document.addEventListener("mousedown", h); document.addEventListener("keydown", k);
    return () => { document.removeEventListener("mousedown", h); document.removeEventListener("keydown", k); };
  }, []);
  const run = async (kind) => {
    setOpen(false); setMsg(""); setBusy(true);
    try {
      if (kind === "pdf") await downloadPDF(trip);
      else if (kind === "html") saveBlob(new Blob([itineraryDocument(trip)], { type: "text/html" }), `${slug(trip.city)}-itinerary.html`);
      else saveBlob(new Blob([trip.markdown], { type: "text/markdown" }), `${slug(trip.city)}-itinerary.md`);
    } catch (e) { setMsg(e.message || "Download failed. Try the HTML version."); setTimeout(() => setMsg(""), 7000); }
    setBusy(false);
  };
  return (
    <div ref={ref} style={{ position: "relative" }}>
      <button className="btn" onClick={() => setOpen(o => !o)} disabled={busy} aria-haspopup="menu" aria-expanded={open}>{busy ? "Preparing…" : "⬇ Download"}</button>
      {open && (
        <div className="dl-menu" role="menu">
          <button className="dl-item" role="menuitem" onClick={() => run("pdf")}><b>PDF document</b><span>Best for printing and sharing</span></button>
          <button className="dl-item" role="menuitem" onClick={() => run("html")}><b>Web page (.html)</b><span>Opens in any browser, keeps the layout</span></button>
          <button className="dl-item" role="menuitem" onClick={() => run("md")}><b>Markdown (.md)</b><span>Plain text for notes apps</span></button>
        </div>
      )}
      {msg && <div className="dl-err">{msg}</div>}
    </div>
  );
};

const PlanLoading = ({ elapsed }) => {
  const stage = elapsed < 12 ? "Researching well-known places for your interests…"
    : elapsed < 35 ? "Checking entry fees, meal prices and distances…"
    : elapsed < 70 ? "Laying out each day and balancing the budget…"
    : "Almost there. Longer trips take a little more time.";
  return (
    <div style={{ maxWidth: 760 }}>
      <div style={{ color: "var(--muted)", fontSize: ".86rem", marginBottom: 16 }}><span className="spinner"/>&nbsp; {stage} <span style={{ opacity: .7 }}>{elapsed}s</span></div>
      <div className="sk" style={{ height: 34, width: "55%", marginBottom: 12 }}/>
      <div className="sk" style={{ height: 14, width: "80%", marginBottom: 22 }}/>
      <div className="sk" style={{ height: 120, marginBottom: 14 }}/>
      <div style={{ display: "grid", gridTemplateColumns: "repeat(auto-fill,minmax(220px,1fr))", gap: 10 }}>
        {[0, 1, 2].map(i => <div key={i} className="sk" style={{ height: 82 }}/>)}
      </div>
    </div>
  );
};

const TripView = ({ trip }) => {
  const it = trip.itinerary;
  const [tab, setTab] = useState("overview");
  const [copied, setCopied] = useState(false);
  useEffect(() => { setTab("overview"); }, [trip.id]);
  const cur = trip.currency;
  const day = typeof tab === "number" && it ? it.days.find(d => d.day === tab) : null;

  const header = (
    <div className="it-head">
      <div style={{ flex: 1, minWidth: 240 }}>
        <div className="it-title">{(it && it.title) || trip.city}</div>
        <div className="it-meta">
          <span className="pill">📍 {trip.city}</span>
          <span className="pill">{trip.days} {trip.days === 1 ? "day" : "days"}</span>
          <span className="pill">{money(trip.budget, cur)}</span>
          {(trip.interests || []).map(i => <span className="pill" key={i}>{i}</span>)}
        </div>
      </div>
      <div className="it-actions">
        <button className="btn ghost" onClick={() => { setCopied(copyText(trip.markdown)); setTimeout(() => setCopied(false), 1800); }}>{copied ? "Copied ✓" : "Copy"}</button>
        <DownloadMenu trip={trip}/>
      </div>
    </div>
  );

  if (!it) {   // older saved trips / AI fallback: plain markdown
    return (<div style={{ maxWidth: 860 }}>{header}<div className="md" dangerouslySetInnerHTML={{ __html: renderMarkdown(trip.markdown) }}/></div>);
  }
  return (
    <div style={{ maxWidth: 880 }}>
      {header}
      <div className="it-tabs" role="tablist">
        <button role="tab" aria-selected={tab === "overview"} className={"it-tab" + (tab === "overview" ? " on" : "")} onClick={() => setTab("overview")}><b>Overview</b><span>Budget & summary</span></button>
        {it.days.map(d => (
          <button role="tab" aria-selected={tab === d.day} key={d.day} className={"it-tab" + (tab === d.day ? " on" : "")} onClick={() => setTab(d.day)}>
            <b>Day {d.day}</b><span>{d.theme || d.city || money(d.subtotal, cur)}</span>
          </button>
        ))}
      </div>
      {day ? <DayView d={day} cur={cur}/> : <Overview trip={trip} it={it} onOpen={setTab}/>}
      {trip.notice && <div className="callout">⚠️ {trip.notice}</div>}
    </div>
  );
};

const Planner = ({ place }) => {
  const asDest = (p) => ({ name: p.name, label: p.label });
  const [dests, setDests] = useState(place ? [asDest(place)] : []);
  const [budget, setBudget] = useState("");
  const [currency, setCurrency] = useState(CUR_BY_COUNTRY[place?.country_code] || "USD");
  const [days, setDays] = useState(3);
  const [interests, setInterests] = useState("food, heritage");
  const [loading, setLoading] = useState(false);
  const [elapsed, setElapsed] = useState(0);
  const [error, setError] = useState("");
  const [result, setResult] = useState(null);
  const [saved, setSaved] = useState(() => { try { return JSON.parse(lsGet("saferoam_trips", "[]")); } catch (e) { return []; } });

  useEffect(() => { if (place) { setDests([asDest(place)]); setCurrency(c => CUR_BY_COUNTRY[place.country_code] || c); } }, [place?.label]);
  useEffect(() => {
    if (!loading) return;
    setElapsed(0);
    const t = setInterval(() => setElapsed(e => e + 1), 1000);
    return () => clearInterval(t);
  }, [loading]);

  const list = interests.split(",").map(s => s.trim().toLowerCase()).filter(Boolean);
  const toggle = (c) => setInterests((list.includes(c) ? list.filter(x => x !== c) : [...list, c]).join(", "));
  const addDest = (d) => setDests(cur => cur.length >= 4 || cur.some(x => x.label.toLowerCase() === d.label.toLowerCase()) ? cur : [...cur, d]);
  const cityStr = dests.length === 1 ? dests[0].label : dests.map(d => d.name).join(", ");
  const perDay = Number(budget) > 0 && days > 0 ? Math.round(Number(budget) / days) : 0;

  const persist = (next) => { setSaved(next); lsSet("saferoam_trips", JSON.stringify(next)); };
  const generate = async () => {
    setError(""); setResult(null); setLoading(true);
    try {
      const r = await runJob("api/itinerary", { city: cityStr, budget: Number(budget), currency, days: Number(days), interests: list });
      const trip = { ...r, id: Date.now(), createdAt: new Date().toLocaleString() };
      setResult(trip);
      persist([trip, ...saved].slice(0, 10));
    } catch (e) { setError(e.message); }
    setLoading(false);
  };
  const remove = (id) => { persist(saved.filter(t => t.id !== id)); if (result && result.id === id) setResult(null); };

  return (
    <div style={{ display: "grid", gridTemplateColumns: "340px 1fr", flex: 1, overflow: "hidden" }}>
      <div style={{ background: "var(--surface)", borderRight: "1px solid var(--border)", overflowY: "auto", padding: "16px 18px" }}>
        <div className="sect">Plan a budget trip</div>
        <div className="field"><label>Where to?</label>
          {dests.length > 0 && (
            <div className="dest-tags">
              {dests.map((d, i) => (
                <span className="dest-tag" key={d.label + i}>📍 {d.name}
                  <button aria-label={`Remove ${d.name}`} onClick={() => setDests(cur => cur.filter((_, j) => j !== i))}>✕</button></span>
              ))}
            </div>
          )}
          <PlaceSearch square clearOnPick placeholder={dests.length ? "Add another destination…" : "Search a city, e.g. Jaipur"}
            onPick={s => addDest({ name: s.name, label: s.label })} onSubmitText={t => addDest({ name: t, label: t })}/>
          <div style={{ fontSize: ".7rem", color: "var(--muted)", marginTop: 2 }}>Pick suggestions from the list. Add up to 4 for a multi-city trip.</div>
        </div>
        <div style={{ display: "grid", gridTemplateColumns: "1fr 90px", gap: 8 }}>
          <div className="field"><label>Total budget</label>
            <input className="inp" type="number" min="1" value={budget} onChange={e => setBudget(e.target.value)} placeholder="e.g. 20000"/></div>
          <div className="field"><label>Currency</label>
            <select className="inp" value={currency} onChange={e => setCurrency(e.target.value)}>{CURRENCIES.map(c => <option key={c}>{c}</option>)}</select></div>
        </div>
        <div className="field"><label>Trip length</label>
          <div className="stepper">
            <button aria-label="Fewer days" onClick={() => setDays(d => Math.max(1, d - 1))} disabled={days <= 1}>−</button>
            <span>{days} {days === 1 ? "day" : "days"}</span>
            <button aria-label="More days" onClick={() => setDays(d => Math.min(14, d + 1))} disabled={days >= 14}>+</button>
          </div>
          {perDay > 0 && <div style={{ fontSize: ".7rem", color: "var(--muted)", marginTop: 2 }}>About {money(perDay, currency)} per day</div>}
        </div>
        <div className="field"><label>Interests</label>
          <input className="inp" value={interests} onChange={e => setInterests(e.target.value)} placeholder="food, nature, nightlife"/>
          <div style={{ display: "flex", flexWrap: "wrap", gap: 5, marginTop: 4 }}>
            {INTEREST_CHIPS.map(c => <span key={c} className={"chip" + (list.includes(c) ? " on" : "")} onClick={() => toggle(c)}>{c}</span>)}
          </div>
        </div>
        <button className="btn" style={{ width: "100%", marginTop: 4 }} onClick={generate} disabled={loading || !dests.length || !budget || !list.length}>
          {loading ? "Planning…" : "✨ Generate itinerary"}
        </button>
        <div style={{ fontSize: ".7rem", color: "var(--muted)", marginTop: 8, lineHeight: 1.4 }}>Budget covers lodging, food, local transport and activities (not flights).</div>

        <div className="sect" style={{ marginTop: 22 }}>Saved trips</div>
        {saved.length === 0 && <div style={{ fontSize: ".78rem", color: "var(--muted)" }}>Generated itineraries are saved here (in this browser).</div>}
        {saved.map(t => (
          <div key={t.id} onClick={() => { setError(""); setResult(t); }} style={{ background: "var(--card-bg)", border: `1px solid ${result && result.id === t.id ? "var(--accent)" : "var(--border)"}`, borderRadius: 10, padding: "9px 12px", marginBottom: 7, cursor: "pointer", display: "flex", alignItems: "center", gap: 8 }}>
            <div style={{ flex: 1, minWidth: 0 }}>
              <div style={{ fontSize: ".84rem", fontWeight: 600, whiteSpace: "nowrap", overflow: "hidden", textOverflow: "ellipsis" }}>{t.city}</div>
              <div style={{ fontSize: ".7rem", color: "var(--muted)" }}>{t.days}d · {t.budget} {t.currency} · {t.createdAt}</div>
            </div>
            <span title="Delete" onClick={(e) => { e.stopPropagation(); remove(t.id); }} style={{ color: "var(--muted)", fontSize: ".9rem", padding: "0 4px" }}>✕</span>
          </div>
        ))}
      </div>

      <div style={{ overflowY: "auto", padding: "18px 26px 40px", background: "var(--bg)" }}>
        {loading && <PlanLoading elapsed={elapsed}/>}
        {error && !loading && <div style={{ color: "var(--accent2)", fontSize: ".85rem" }}>⚠️ {error}</div>}
        {!loading && !error && !result && (
          <div style={{ color: "var(--muted)", fontSize: ".88rem", maxWidth: 520, lineHeight: 1.6 }}>
            <div style={{ fontFamily: "'Playfair Display',serif", fontSize: "1.3rem", color: "var(--accent)", marginBottom: 8 }}>Your trips</div>
            Choose a destination, set a budget and generate a day-by-day plan: a timeline for every day, realistic per-person costs, distance-based transport fares and a full budget breakdown. Download it as a PDF when you're done.
          </div>
        )}
        {result && !loading && <TripView trip={result}/>}
      </div>
    </div>
  );
};

// ── COMMUNITY ─────────────────────────────────────────────────────────────────
const CAT = { tip: ["💡 Tip", "#f0a500"], warning: ["⚠️ Warning", "#e05c5c"], food: ["🍜 Food", "#3fb950"], stay: ["🏨 Stay", "#60a5fa"] };

const Community = ({ place }) => {
  const city = place ? place.name : "";
  const [posts, setPosts] = useState([]);
  const [author, setAuthor] = useState(lsGet("saferoam_author", ""));
  const [text, setText] = useState("");
  const [cat, setCat] = useState("tip");
  const [err, setErr] = useState("");
  const load = () => getJSON(`api/community?city=${encodeURIComponent(city)}`).then(r => setPosts(r.posts)).catch(() => {});
  useEffect(() => { if (city) load(); }, [city]);

  const submit = async () => {
    setErr("");
    try {
      await post("api/community", { city, author, text, category: cat });
      lsSet("saferoam_author", author); setText(""); load();
    } catch (e) { setErr(e.message); }
  };

  return (
    <div style={{flex:1, overflowY:"auto", padding:"20px 28px", background:"var(--bg)"}}>
      <div style={{maxWidth:760, margin:"0 auto"}}>
        <div style={{fontFamily:"'Playfair Display',serif", fontSize:"1.35rem", color:"var(--accent)", marginBottom:4}}>Community · {place ? place.label : ""}</div>
        <div style={{fontSize:"0.8rem", color:"var(--muted)", marginBottom:16}}>Share tips and warnings with other travellers. Posts are stored by this app's server.</div>
        <div style={{background:"var(--card-bg)", border:"1px solid var(--border)", borderRadius:12, padding:14, marginBottom:18}}>
          <div style={{display:"flex", gap:8, marginBottom:8}}>
            <input className="inp" style={{flex:1}} placeholder="Your name (optional)" value={author} onChange={e => setAuthor(e.target.value)} maxLength={30}/>
            <select className="inp" style={{width:140}} value={cat} onChange={e => setCat(e.target.value)}>
              {Object.entries(CAT).map(([k, v]) => <option key={k} value={k}>{v[0]}</option>)}</select>
          </div>
          <textarea className="inp" rows={3} placeholder={`Share a tip about ${city}…`} value={text} onChange={e => setText(e.target.value)} maxLength={400} style={{resize:"vertical"}}/>
          <div style={{display:"flex", alignItems:"center", marginTop:8}}>
            <span style={{fontSize:"0.76rem", color:"var(--accent2)"}}>{err}</span>
            <button className="btn" style={{marginLeft:"auto"}} onClick={submit} disabled={text.trim().length < 3}>Post</button>
          </div>
        </div>
        {posts.length === 0 && <div style={{color:"var(--muted)", fontSize:"0.84rem"}}>No posts for {city} yet — be the first!</div>}
        {posts.map(p => (
          <div key={p.id} style={{background:"var(--card-bg)", borderLeft:`3px solid ${CAT[p.category][1]}`, borderRadius:10, padding:"10px 14px", marginBottom:9, animation:"fadeIn .25s ease"}}>
            <div style={{display:"flex", gap:8, fontSize:"0.74rem", color:"var(--muted)", marginBottom:4}}>
              <span style={{color:CAT[p.category][1], fontWeight:600}}>{CAT[p.category][0]}</span>
              <span>{p.author}</span><span style={{marginLeft:"auto"}}>{new Date(p.ts * 1000).toLocaleDateString()}</span>
            </div>
            <div style={{fontSize:"0.84rem", lineHeight:1.5, whiteSpace:"pre-wrap"}}>{p.text}</div>
          </div>
        ))}
      </div>
    </div>
  );
};

// ── SOS ───────────────────────────────────────────────────────────────────────
const SOSModal = ({ onClose, place, safetyInfo, hospitals, userLoc }) => {
  const [contact, setContact] = useState(lsGet("saferoam_contact", ""));
  const [copied, setCopied] = useState(false);
  const em = (safetyInfo && safetyInfo.emergency) || {};
  const loc = userLoc ? `https://www.google.com/maps?q=${userLoc.lat},${userLoc.lon}` : null;
  const msg = "🚨 SOS from SafeRoam — I need help. " + (loc ? `My location: ${loc}` : `I'm in ${place ? place.label : "an unknown location"}.`);
  const labels = { police: "👮 Police", ambulance: "🚑 Ambulance", fire: "🚒 Fire", tourist: "🧭 Tourist helpline" };
  const digits = contact.replace(/[^\d+]/g, "");
  const linkBtn = { textDecoration:"none", display:"inline-block", textAlign:"center", fontSize:"0.78rem" };

  return (
    <div onClick={onClose} style={{ position:"fixed", inset:0, background:"rgba(0,0,0,0.6)", display:"flex", alignItems:"center", justifyContent:"center", zIndex:999, animation:"fadeIn 0.2s ease" }}>
      <div onClick={e => e.stopPropagation()} style={{ background:"#1a1013", border:"2px solid var(--accent2)", borderRadius:16, padding:"22px 26px", width:440, maxWidth:"94vw", maxHeight:"92vh", overflowY:"auto", boxShadow:"0 0 40px rgba(224,92,92,0.4)" }}>
        <div style={{display:"flex", alignItems:"center", marginBottom:4}}>
          <div style={{fontSize:"1.1rem", fontWeight:700}}>🚨 Emergency help</div>
          <span onClick={onClose} style={{marginLeft:"auto", cursor:"pointer", color:"var(--muted)", fontSize:"1.1rem"}}>✕</span>
        </div>
        <div style={{fontSize:"0.76rem", color:"#fca5a5", marginBottom:14}}>SafeRoam can't call anyone for you — tap a number to dial, or share your location below.</div>

        {Object.keys(labels).filter(k => em[k]).map(k => (
          <div key={k} style={{display:"flex", alignItems:"center", background:"var(--surface2)", borderRadius:10, padding:"9px 12px", marginBottom:7}}>
            <span style={{fontSize:"0.85rem"}}>{labels[k]}</span>
            <a href={`tel:${firstNumber(em[k])}`} style={{marginLeft:"auto", fontWeight:700, color:"#fff", textDecoration:"none", background:"var(--accent2)", borderRadius:16, padding:"4px 14px", fontSize:"0.85rem"}}>{em[k]}</a>
          </div>
        ))}
        {!Object.keys(em).length && <div style={{fontSize:"0.8rem", color:"var(--muted)", marginBottom:10}}>Emergency numbers for this city haven't loaded. Dial the local emergency number (112 / 911 / 999 depending on the country).</div>}

        {hospitals.length > 0 && (<>
          <div className="sect" style={{marginTop:12}}>Nearest hospitals</div>
          {hospitals.map((h, i) => (
            <a key={i} href={`https://www.google.com/maps/search/?api=1&query=${h.lat},${h.lon}`} target="_blank" rel="noopener noreferrer"
               style={{display:"flex", background:"var(--surface2)", borderRadius:10, padding:"8px 12px", marginBottom:6, textDecoration:"none", color:"var(--text)", fontSize:"0.8rem"}}>
              <span>🏥 {h.name}</span><span style={{marginLeft:"auto", color:"var(--muted)"}}>{h.dist} km</span>
            </a>
          ))}
        </>)}

        <div className="sect" style={{marginTop:14}}>Share my location</div>
        <div style={{fontSize:"0.76rem", color:"var(--muted)", marginBottom:8}}>
          {loc ? <>📍 Location found · <a href={loc} target="_blank" rel="noopener noreferrer">open in Maps</a></> : "Location permission not granted — the message will only mention the city."}
        </div>
        <div className="field" style={{marginBottom:8}}><label>Emergency contact (with country code)</label>
          <input className="inp" value={contact} placeholder="+91 98xxxxxxxx" onChange={e => setContact(e.target.value)} onBlur={() => lsSet("saferoam_contact", contact)}/></div>
        <div style={{display:"flex", flexWrap:"wrap", gap:6}}>
          <button className="btn danger" onClick={() => { setCopied(copyText(msg)); setTimeout(() => setCopied(false), 2000); }}>{copied ? "Copied ✓" : "Copy SOS message"}</button>
          {digits && <a className="btn ghost" style={linkBtn} href={`sms:${digits}?body=${encodeURIComponent(msg)}`}>SMS contact</a>}
          {digits && <a className="btn ghost" style={linkBtn} target="_blank" rel="noopener noreferrer" href={`https://wa.me/${digits.replace("+", "")}?text=${encodeURIComponent(msg)}`}>WhatsApp</a>}
        </div>
      </div>
    </div>
  );
};

// ── MAIN APP ──────────────────────────────────────────────────────────────────
const App = () => {
  const [search, setSearch] = useState("");
  const [selected, setSelected] = useState(null);
  const [activeNav, setActiveNav] = useState("Dashboard");
  const [sosOpen, setSosOpen] = useState(false);

  const [place, setPlace] = useState(null);
  const [places, setPlaces] = useState([]);
  const [placesLoading, setPlacesLoading] = useState(true);
  const [placesErr, setPlacesErr] = useState("");
  const [weather, setWeather] = useState(null);
  const [safetyInfo, setSafetyInfo] = useState(null);
  const [hospitals, setHospitals] = useState([]);
  const [userLoc, setUserLoc] = useState(null);
  const [pendingAsk, setPendingAsk] = useState(null);
  const tok = useRef(0);

  const askGeo = () => {
    if (!navigator.geolocation) return;
    navigator.geolocation.getCurrentPosition(
      p => setUserLoc({ lat: p.coords.latitude, lon: p.coords.longitude }), () => {}, { timeout: 8000, maximumAge: 600000 });
  };

  const loadCity = async (arg) => {   // arg: a place picked from the suggestions, or free text
    const my = ++tok.current;
    setPlacesErr(""); setPlacesLoading(true);
    let p;
    if (arg && typeof arg === "object") p = arg;
    else {
      try { p = await post("api/geocode", { query: arg }); }
      catch (e) { if (my === tok.current) { setPlacesErr(e.message); setPlacesLoading(false); } return; }
    }
    if (my !== tok.current) return;
    setPlace(p); setSelected(null); setPlaces([]); setWeather(null); setSafetyInfo(null); setHospitals([]); setSearch("");
    getJSON(`api/weather?lat=${p.lat}&lon=${p.lon}`).then(w => my === tok.current && setWeather(w)).catch(() => my === tok.current && setWeather({ error: true }));
    runJob("api/safety", p).then(s => my === tok.current && setSafetyInfo(s)).catch(() => my === tok.current && setSafetyInfo({ error: true, emergency: {}, alerts: [] }));
    runJob("api/places", p).then(r => my === tok.current && setPlaces(r.places)).catch(e => my === tok.current && setPlacesErr(e.message)).finally(() => my === tok.current && setPlacesLoading(false));
  };

  useEffect(() => { loadCity("Delhi, India"); askGeo(); }, []);

  const userNear = userLoc && place && haversine(userLoc.lat, userLoc.lon, place.lat, place.lon) < 80 ? userLoc : null;
  const hLat = userNear ? userNear.lat : place?.lat, hLon = userNear ? userNear.lon : place?.lon;
  useEffect(() => {
    if (hLat == null) return;
    let live = true;
    getJSON(`api/hospitals?lat=${hLat}&lon=${hLon}`).then(r => live && setHospitals(r.hospitals)).catch(() => {});
    return () => { live = false; };
  }, [hLat, hLon]);

  const filtered = useMemo(() => {
    const s = search.toLowerCase();
    return places.filter(d => d.name.toLowerCase().includes(s) || (d.area || "").toLowerCase().includes(s) || d.tags.some(t => t.toLowerCase().includes(s)));
  }, [places, search]);

  const stats = useMemo(() => {
    const n = places.length;
    const alerts = safetyInfo && safetyInfo.alerts ? safetyInfo.alerts.filter(a => a.type !== "safe").length : 0;
    return {
      score: n ? Math.round(places.reduce((s, d) => s + SAFETY_SCORE[d.safety], 0) / n) + "%" : "—",
      rating: n ? (places.reduce((s, d) => s + d.rating, 0) / n).toFixed(1) + "★" : "—",
      nearby: n ? String(n) : "—", alerts: safetyInfo ? String(alerts) : "—",
    };
  }, [places, safetyInfo]);

  const alertCards = useMemo(() => {
    const icon = { safe: "✅", moderate: "⚡", caution: "⚠️" };
    const a = ((safetyInfo && safetyInfo.alerts) || []).map(x => ({ icon: icon[x.type], title: x.title, desc: x.desc, type: x.type }));
    if (hospitals[0]) a.push({ icon: "🏥", title: `Nearest hospital: ${hospitals[0].dist} km`, desc: hospitals[0].name, type: "moderate" });
    return a;
  }, [safetyInfo, hospitals]);

  const ctx = useMemo(() => ({
    city: place ? place.label : "", selected: selected ? { name: selected.name, area: selected.area, safety: selected.safety } : null,
    weather: weather && !weather.error ? `${weather.temp}°C, ${weather.desc}` : null,
    emergency: safetyInfo ? safetyInfo.emergency : null,
  }), [place, selected, weather, safetyInfo]);

  const handleSOS = () => { askGeo(); setSosOpen(true); };
  const ask = (q) => { setActiveNav("Dashboard"); setPendingAsk({ text: q, n: Date.now() }); };
  const cityName = place ? place.name : "this city";
  const chips = [
    ["🍜 Street Food", `What are the best street food spots in ${cityName} and what should I watch out for?`],
    ["🏛️ Heritage Walk", `Suggest a one-day heritage walk in ${cityName} with a sensible order.`],
    ["🌙 Night Market", `Are there night markets or evening spots in ${cityName} that are safe for tourists?`],
    ["🚇 Metro Route", `How do I get around ${cityName} by public transport as a tourist?`],
    ["📸 Photo Spots", `Best photo spots in ${cityName} and the best time of day for each?`],
    ["🛡️ Safe Areas", `Which areas of ${cityName} are safest for tourists, and which should I avoid at night?`],
    ["🏨 Stay Deals", `Where should a tourist stay in ${cityName}? Safe neighbourhoods and budget tiers.`],
    ["🚑 Hospitals", `Where are the nearest hospitals to me in ${cityName} and who do I call in an emergency?`],
  ];

  const dashboard = (
    <div style={{display:"grid", gridTemplateColumns:"280px 1fr 300px", flex:1, overflow:"hidden"}}>
      {/* LEFT: Destinations */}
      <div style={{ background:"var(--surface)", borderRight:"1px solid var(--border)", display:"flex", flexDirection:"column", overflow:"hidden" }}>
        <div style={{ fontSize:"0.75rem", color:"var(--muted)", textTransform:"uppercase", letterSpacing:"0.1em", padding:"15px 18px 8px", fontWeight:600 }}>Attractions{place ? " · " + place.name : ""}</div>
        <div style={{ margin:"0 14px 10px", display:"flex", alignItems:"center", gap:8, background:"var(--bg)", border:"1px solid var(--border)", borderRadius:8, padding:"7px 12px" }}>
          <span style={{color:"var(--muted)"}}>🔍</span>
          <input value={search} onChange={e => setSearch(e.target.value)} placeholder="Filter these places…"
            style={{ flex:1, background:"none", border:"none", outline:"none", color:"var(--text)", fontSize:"0.83rem" }}/>
        </div>
        <div style={{flex:1, overflowY:"auto", padding:"0 12px 12px"}}>
          {placesLoading ? <div style={{color:"var(--muted)", fontSize:"0.82rem", padding:"16px 4px"}}><span className="spinner"/>&nbsp; Loading places…</div>
            : placesErr && !places.length ? (
              <div style={{color:"var(--accent2)", fontSize:"0.8rem", padding:"12px 4px", lineHeight:1.5}}>
                {placesErr}<div><button className="btn ghost" style={{marginTop:8, padding:"4px 14px"}} onClick={() => loadCity(place || "Delhi, India")}>Retry</button></div></div>)
            : filtered.length === 0 ? <div style={{color:"var(--muted)", fontSize:"0.82rem", padding:"16px 4px"}}>No places found.</div>
            : filtered.map(d => <DestCard key={d.id} dest={d} active={selected?.id === d.id} onClick={() => setSelected(d)}/>)}
        </div>
      </div>

      {/* CENTER: Map + Chat */}
      <div style={{display:"flex", flexDirection:"column", overflow:"hidden", background:"var(--bg)"}}>
        <MapArea place={place} places={places} visible={filtered} selected={selected} onSelect={setSelected} user={userLoc} hospitals={hospitals} onLocate={askGeo} loading={placesLoading} error={placesErr}/>
        {activeNav !== "Safety Map" && <ChatPanel place={place} selected={selected} ctx={ctx} pendingAsk={pendingAsk} onAskDone={() => setPendingAsk(null)}/>}
      </div>

      {/* RIGHT: Live Info */}
      <div style={{ background:"var(--surface)", borderLeft:"1px solid var(--border)", display:"flex", flexDirection:"column", overflowY:"auto", paddingBottom:16 }}>
        <div style={{ fontSize:"0.75rem", color:"var(--muted)", textTransform:"uppercase", letterSpacing:"0.1em", padding:"15px 16px 8px", fontWeight:600 }}>Live Info</div>

        <div style={{padding:"0 14px", marginBottom:10}}>
          <div className="sect">Weather</div>
          <div style={{background:"var(--card-bg)", border:"1px solid var(--border)", borderRadius:10, padding:12}}>
            {weather && !weather.error ? (<>
              <div style={{display:"flex", alignItems:"center", justifyContent:"space-between"}}>
                <div>
                  <div style={{fontFamily:"'Playfair Display',serif", fontSize:"1.9rem", color:"var(--accent)"}}>{weather.temp}°C</div>
                  <div style={{fontSize:"0.76rem", color:"var(--muted)"}}>{weather.desc} · {place?.name}</div>
                </div>
                <div style={{fontSize:"2.2rem"}}>{weather.icon}</div>
              </div>
              <div style={{marginTop:8, display:"flex", gap:12, flexWrap:"wrap"}}>
                <span style={{fontSize:"0.72rem", color:"var(--muted)"}}>💧 {weather.humidity}% Humidity</span>
                <span style={{fontSize:"0.72rem", color:"var(--muted)"}}>💨 {weather.wind} km/h</span>
                <span style={{fontSize:"0.72rem", color:"var(--muted)"}}>🌡️ Feels {weather.feels}°C</span>
              </div>
              <div style={{marginTop:6, fontSize:"0.73rem", color:"var(--accent)"}}>{weather.hot ? "⚠️ " : ""}{weather.advisory}</div>
            </>) : weather ? <div style={{fontSize:"0.78rem", color:"var(--muted)"}}>Weather unavailable right now.</div>
              : <div style={{fontSize:"0.78rem", color:"var(--muted)"}}><span className="spinner"/>&nbsp; Loading weather…</div>}
          </div>
        </div>

        <div style={{padding:"0 14px", marginBottom:10}}>
          <div className="sect">Safety Alerts</div>
          {!safetyInfo && <div style={{fontSize:"0.78rem", color:"var(--muted)"}}><span className="spinner"/>&nbsp; Checking advisories…</div>}
          {safetyInfo && alertCards.length === 0 && <div style={{fontSize:"0.78rem", color:"var(--muted)"}}>No alerts could be loaded.</div>}
          {alertCards.map((a, i) => (
            <div key={i} style={{ background:"var(--card-bg)", borderRadius:10, padding:"10px 12px", marginBottom:7, display:"flex", gap:10, alignItems:"flex-start", borderLeft:`3px solid ${SAFETY_COLOR[a.type]}` }}>
              <div style={{fontSize:"1rem", flexShrink:0}}>{a.icon}</div>
              <div>
                <div style={{fontSize:"0.81rem", fontWeight:600, marginBottom:2}}>{a.title}</div>
                <div style={{fontSize:"0.74rem", color:"var(--muted)", lineHeight:1.4}}>{a.desc}</div>
              </div>
            </div>
          ))}
        </div>

        <div style={{padding:"0 14px", marginBottom:10}}>
          <div className="sect">Quick Stats</div>
          <div style={{display:"grid", gridTemplateColumns:"1fr 1fr", gap:8}}>
            {[{ val:stats.score, lbl:"Safety Score", col:"var(--safe)" }, { val:stats.rating, lbl:"Avg Rating", col:"var(--accent)" },
              { val:stats.nearby, lbl:"Nearby Places", col:"var(--text)" }, { val:stats.alerts, lbl:"Active Alerts", col:"var(--accent2)" }].map((s, i) => (
              <div key={i} style={{ background:"var(--card-bg)", border:"1px solid var(--border)", borderRadius:10, padding:"11px", textAlign:"center" }}>
                <div style={{fontFamily:"'Playfair Display',serif", fontSize:"1.35rem", color:s.col}}>{s.val}</div>
                <div style={{fontSize:"0.69rem", color:"var(--muted)", marginTop:2}}>{s.lbl}</div>
              </div>
            ))}
          </div>
        </div>

        {selected && (
          <div style={{padding:"0 14px", marginBottom:10, animation:"fadeIn 0.3s ease"}}>
            <div className="sect">Selected Destination</div>
            <div style={{ background:"var(--card-bg)", border:`1px solid ${SAFETY_COLOR[selected.safety]}44`, borderRadius:10, padding:12 }}>
              <div style={{fontWeight:700, fontSize:"0.95rem", marginBottom:4}}>{selected.name}</div>
              <SafetyBadge safety={selected.safety}/>
              <div style={{marginTop:8, fontSize:"0.75rem", color:"var(--muted)"}}>📍 {selected.area || place?.name} &nbsp;·&nbsp; ★ {selected.rating}</div>
              <div style={{marginTop:6, display:"flex", flexWrap:"wrap", gap:3}}>
                {selected.tags.map(t => (<span key={t} style={{fontSize:"0.68rem", color:"var(--muted)", background:"#21262d", padding:"2px 7px", borderRadius:8}}>{t}</span>))}
              </div>
              <a href={`https://www.google.com/maps/search/?api=1&query=${selected.lat},${selected.lon}`} target="_blank" rel="noopener noreferrer" style={{display:"inline-block", marginTop:8, fontSize:"0.74rem"}}>Open in Google Maps ↗</a>
            </div>
          </div>
        )}

        <div style={{padding:"0 14px"}}>
          <div className="sect">AI Recommendations</div>
          <div style={{display:"flex", flexWrap:"wrap", gap:4}}>
            {chips.map(([label, q]) => (<span key={label} className="chip" style={{marginBottom:2}} onClick={() => ask(q)}>{label}</span>))}
          </div>
        </div>
      </div>
    </div>
  );

  return (
    <div style={{display:"flex", flexDirection:"column", height:"100vh", overflow:"hidden"}}>
      {/* NAVBAR */}
      <div style={{ display:"flex", alignItems:"center", justifyContent:"space-between", padding:"11px 24px", background:"var(--surface)", borderBottom:"1px solid var(--border)", flexShrink:0 }}>
        <div style={{fontFamily:"'Playfair Display',serif", fontSize:"1.3rem", color:"var(--accent)"}}>
          Safe<span style={{color:"var(--text)"}}>Roam</span>
          <span style={{color:"var(--muted)", fontSize:"0.7rem", fontFamily:"'DM Sans',sans-serif", fontWeight:300, marginLeft:10}}>AI Tourist & Safety System</span>
        </div>
        <div style={{display:"flex", gap:5}}>
          {["Dashboard","My Trips","Safety Map","Community"].map(tab => (
            <button key={tab} onClick={() => setActiveNav(tab)} style={{
              padding:"5px 15px", borderRadius:20, fontSize:"0.8rem", fontWeight:500, cursor:"pointer", border:"1px solid transparent", transition:"all 0.18s",
              background: activeNav === tab ? "var(--accent)" : "none", color: activeNav === tab ? "#000" : "var(--muted)",
              borderColor: activeNav === tab ? "var(--accent)" : "transparent" }}>{tab}</button>
          ))}
        </div>
        <div style={{display:"flex", alignItems:"center", gap:12}}>
          <div style={{width:270}}>
            <PlaceSearch value={place ? place.label : ""} placeholder="Search any city…" selectAllOnFocus
              onPick={s => loadCity(s)} onSubmitText={t => loadCity(t)}/>
          </div>
          <button onClick={handleSOS} style={{ background:"var(--accent2)", color:"#fff", border:"none", padding:"5px 16px", borderRadius:20, fontSize:"0.8rem", fontWeight:700, animation:"pulse-sos 2s infinite" }}>🆘 SOS</button>
          <div style={{ width:32, height:32, borderRadius:"50%", background:"linear-gradient(135deg,#f0a500,#e05c5c)", display:"flex", alignItems:"center", justifyContent:"center", fontSize:"0.85rem" }}>👤</div>
        </div>
      </div>

      {activeNav === "My Trips" ? <Planner place={place}/> : activeNav === "Community" ? <Community place={place}/> : dashboard}

      {sosOpen && <SOSModal onClose={() => setSosOpen(false)} place={place} safetyInfo={safetyInfo} hospitals={hospitals} userLoc={userNear}/>}
    </div>
  );
};

ReactDOM.createRoot(document.getElementById("root")).render(<App/>);
</script>
</body>
</html>'''


# ------------------------------------------------------------------- launch --
def launch():
    from werkzeug.serving import make_server
    old = globals().get("_SR_SERVER")
    if old is not None:  # re-running the cell: free the port first
        try:
            old.shutdown()
        except Exception:
            pass
        time.sleep(0.5)
    server = make_server("0.0.0.0", PORT, app, threaded=True)
    globals()["_SR_SERVER"] = server
    threading.Thread(target=server.serve_forever, daemon=True).start()

    try:
        from google.colab import output
        print("SafeRoam is running. If the app below is blank, open it in its own tab (better for GPS/location):")
        try:
            print(output.eval_js(f"google.colab.kernel.proxyPort({PORT})"))
        except Exception:
            pass
        output.serve_kernel_port_as_iframe(PORT, height="900")
    except ImportError:
        print(f"SafeRoam is running at http://localhost:{PORT}")


launch()


Enter your Groq API key: ··········
SafeRoam is running. If the app below is blank, open it in its own tab (better for GPS/location):
https://5055-gpu-t4-s-usw1b1-1mhddaah5cxnh-b.us-west1.prod.r.colab.dev


<IPython.core.display.Javascript object>